# Build Manifest


In [ ]:
#!/usr/bin/env python3
"""
build_manifest.py
=================
Step 0 of the publication-grade pipeline: turn a pile of merged ASD facial-emotion
images into an auditable manifest with (a) exact + near-duplicate flags, (b) inferred
subject/identity groups, and (c) source-dataset provenance.

WHY THIS FILE EXISTS
--------------------
Your current protocol splits/folds at the IMAGE level. When four face datasets are
merged, the same child almost certainly appears in many frames. Image-level splitting
puts frames of the same child in both train and test, which inflates every number you
reported. A Q1 biomedical reviewer will ask for subject-independent evaluation, and if
you cannot produce it the paper is rejected. This script produces the `group` column
that makes subject-independent evaluation possible.

It also produces the `source` column so you can run the two experiments reviewers
always ask for on merged corpora:
  1. Source-probe: can a classifier predict which dataset an image came from?
     (High AUC => your emotion model may be exploiting dataset artifacts.)
  2. Leave-one-dataset-out (LODO) generalization.

INPUT LAYOUT (flexible)
-----------------------
Pass one or more roots as `name=path`, each containing class subfolders:

    dataset_nora/anger/*.jpg
    dataset_nora/fear/*.jpg
    ...

USAGE
-----
    python build_manifest.py \
        --roots nora=/kaggle/input/nora_mendeley \
                ferac=/kaggle/input/ferac \
                talaat=/kaggle/input/talaat \
                hasibur=/kaggle/input/hasibur \
        --out manifest.csv \
        --phash-threshold 6 \
        --identity-threshold 0.55

DEPENDENCIES
------------
    pip install pillow imagehash numpy pandas scikit-learn tqdm
    pip install facenet-pytorch          # optional but STRONGLY recommended
"""

from __future__ import annotations

import argparse
import hashlib
import json
import os
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
CANONICAL_CLASSES = ["anger", "fear", "joy", "natural", "sadness", "surprise"]

# Map the label spellings that appear across the four source datasets onto one scheme.
# EXTEND THIS as you inspect your folders -- an unmapped folder name raises an error
# rather than silently creating a 7th class.
LABEL_ALIASES = {
    "anger": "anger", "angry": "anger", "ang": "anger",
    "fear": "fear", "fearful": "fear", "afraid": "fear",
    "joy": "joy", "happy": "joy", "happiness": "joy", "smile": "joy",
    "natural": "natural", "neutral": "natural", "normal": "natural", "calm": "natural",
    "sadness": "sadness", "sad": "sadness",
    "surprise": "surprise", "surprised": "surprise", "surprize": "surprise",
}


# --------------------------------------------------------------------------------------
# 1. Scan
# --------------------------------------------------------------------------------------
def scan_roots(roots: dict[str, str]) -> pd.DataFrame:
    rows = []
    unmapped = set()
    for source, root in roots.items():
        root = Path(root)
        if not root.exists():
            raise FileNotFoundError(f"root '{source}' -> {root} does not exist")
        for p in sorted(root.rglob("*")):
            if p.suffix.lower() not in IMG_EXT or not p.is_file():
                continue
            raw_label = p.parent.name.strip().lower().replace(" ", "_")
            label = LABEL_ALIASES.get(raw_label)
            if label is None:
                unmapped.add(f"{source}:{raw_label}")
                continue
            rows.append({"path": str(p), "source": source,
                         "raw_label": raw_label, "label": label})
    if unmapped:
        raise ValueError(
            "Unmapped folder names found. Add them to LABEL_ALIASES (or exclude the "
            f"folders) before continuing:\n  " + "\n  ".join(sorted(unmapped))
        )
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("No images found. Check --roots.")
    return df


# --------------------------------------------------------------------------------------
# 2. Duplicate detection (exact + perceptual)
# --------------------------------------------------------------------------------------
def md5_of(path: str, chunk: int = 1 << 20) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        while (b := f.read(chunk)):
            h.update(b)
    return h.hexdigest()


def compute_hashes(df: pd.DataFrame) -> pd.DataFrame:
    try:
        import imagehash
    except ImportError:
        sys.exit("pip install imagehash")

    md5s, phashes, sizes, bad = [], [], [], []
    for path in tqdm(df["path"], desc="hashing"):
        try:
            with Image.open(path) as im:
                im = im.convert("RGB")
                sizes.append(f"{im.width}x{im.height}")
                phashes.append(str(imagehash.phash(im, hash_size=8)))
            md5s.append(md5_of(path))
        except Exception as e:  # corrupt file
            bad.append((path, repr(e)))
            sizes.append(""); phashes.append(""); md5s.append("")
    df = df.copy()
    df["md5"] = md5s
    df["phash"] = phashes
    df["resolution"] = sizes
    if bad:
        print(f"[warn] {len(bad)} unreadable images dropped", file=sys.stderr)
        df = df[df["md5"] != ""].reset_index(drop=True)
    return df


def _hex_to_bits(h: str) -> np.ndarray:
    return np.array([int(b) for b in bin(int(h, 16))[2:].zfill(len(h) * 4)], dtype=np.uint8)


def flag_duplicates(df: pd.DataFrame, threshold: int = 6) -> pd.DataFrame:
    """Exact (md5) and near (pHash Hamming <= threshold) duplicate flagging.

    Near-duplicates get a shared `dup_cluster`. Keep ONE representative per cluster
    for training, but never let two members of a cluster land in different CV folds.
    """
    df = df.copy()

    # exact
    df["is_exact_dup"] = df.duplicated(subset=["md5"], keep="first")

    # near: bucket by the first 16 bits to keep pairwise cost sane, then exact Hamming.
    bits = np.stack([_hex_to_bits(h) for h in df["phash"]])
    n = len(df)
    parent = list(range(n))

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[max(ra, rb)] = min(ra, rb)

    buckets = defaultdict(list)
    for i in range(n):
        # multi-probe: 4 rotated 16-bit prefixes so a few flipped bits do not hide a pair
        for shift in (0, 16, 32, 48):
            key = (shift, bits[i, shift:shift + 16].tobytes())
            buckets[key].append(i)

    for idxs in tqdm(buckets.values(), desc="near-dup"):
        if len(idxs) < 2 or len(idxs) > 4000:
            continue
        sub = bits[idxs]
        d = (sub[:, None, :] != sub[None, :, :]).sum(-1)
        ii, jj = np.where(np.triu(d <= threshold, k=1))
        for a, b in zip(ii, jj):
            union(idxs[a], idxs[b])

    df["dup_cluster"] = [f"D{find(i):06d}" for i in range(n)]
    sizes = df["dup_cluster"].value_counts()
    df["dup_cluster_size"] = df["dup_cluster"].map(sizes)
    return df


# --------------------------------------------------------------------------------------
# 3. Identity (subject) clustering  -- the column that makes your splits defensible
# --------------------------------------------------------------------------------------
def infer_identities(df: pd.DataFrame, threshold: float = 0.55,
                     batch_size: int = 64, device: str = "cuda") -> pd.DataFrame:
    """Cluster faces by identity using FaceNet embeddings + agglomerative clustering.

    `threshold` is a cosine distance cut-off. 0.5-0.6 is the usual operating range for
    VGGFace2-trained InceptionResnetV1. TUNE IT: over-merging (too high) throws away
    data by making groups huge; under-merging (too low) leaves subject leakage.

    Validate the choice by eyeballing ~30 random clusters and reporting, in the paper,
    the purity you observed. Reviewers accept inferred identities if you show the
    validation; they do not accept unvalidated ones.
    """
    try:
        import torch
        from facenet_pytorch import InceptionResnetV1
    except ImportError:
        print("[warn] facenet-pytorch not installed -- falling back to dup_cluster as "
              "the grouping key. This is WEAKER: it removes duplicate leakage but NOT "
              "subject leakage. Install facenet-pytorch before submitting.",
              file=sys.stderr)
        df = df.copy()
        df["group"] = df["dup_cluster"]
        df["group_source"] = "dup_cluster_fallback"
        return df

    from sklearn.cluster import AgglomerativeClustering
    from sklearn.preprocessing import normalize

    device = device if torch.cuda.is_available() else "cpu"
    net = InceptionResnetV1(pretrained="vggface2").eval().to(device)

    embs, ok_idx = [], []
    buf, buf_idx = [], []

    def flush():
        if not buf:
            return
        x = torch.stack(buf).to(device)
        with torch.no_grad():
            e = net(x).cpu().numpy()
        embs.append(e)
        ok_idx.extend(buf_idx)
        buf.clear(); buf_idx.clear()

    for i, path in enumerate(tqdm(df["path"], desc="embedding")):
        try:
            with Image.open(path) as im:
                im = im.convert("RGB").resize((160, 160))
            t = torch.from_numpy(np.asarray(im)).permute(2, 0, 1).float()
            t = (t - 127.5) / 128.0
            buf.append(t); buf_idx.append(i)
        except Exception:
            continue
        if len(buf) == batch_size:
            flush()
    flush()

    E = normalize(np.concatenate(embs, 0))
    clust = AgglomerativeClustering(
        n_clusters=None, distance_threshold=threshold,
        metric="cosine", linkage="average",
    ).fit(E)

    df = df.copy()
    df["group"] = [f"UNK{i}" for i in range(len(df))]
    df.loc[df.index[ok_idx], "group"] = [f"ID{c:05d}" for c in clust.labels_]
    df["group_source"] = "facenet_agglomerative"

    # A duplicate cluster must never straddle two identity groups.
    for dc, sub in df.groupby("dup_cluster"):
        if sub["group"].nunique() > 1:
            df.loc[sub.index, "group"] = sub["group"].iloc[0]
    return df


# --------------------------------------------------------------------------------------
# 4. Audit report
# --------------------------------------------------------------------------------------
def audit(df: pd.DataFrame) -> dict:
    g = df.groupby("group")
    rep = {
        "n_images": int(len(df)),
        "n_exact_duplicates": int(df["is_exact_dup"].sum()),
        "n_near_dup_clusters_gt1": int((df["dup_cluster_size"] > 1).sum()),
        "n_groups": int(df["group"].nunique()),
        "images_per_group_mean": float(g.size().mean()),
        "images_per_group_max": int(g.size().max()),
        "class_counts": df["label"].value_counts().to_dict(),
        "source_counts": df["source"].value_counts().to_dict(),
        "class_x_source": df.pivot_table(index="label", columns="source",
                                         values="path", aggfunc="count").fillna(0)
                            .astype(int).to_dict(),
        # groups that span >1 source are a red flag: same child in two datasets
        "groups_spanning_sources": int(
            (g["source"].nunique() > 1).sum()),
        # a group that spans >1 label is either a clustering error or a labelling
        # inconsistency between source datasets -- both must be discussed in the paper
        "groups_spanning_labels": int((g["label"].nunique() > 1).sum()),
    }
    return rep


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--roots", nargs="+", required=True,
                    help="source_name=path pairs")
    ap.add_argument("--out", default="manifest.csv")
    ap.add_argument("--audit-out", default="manifest_audit.json")
    ap.add_argument("--phash-threshold", type=int, default=6)
    ap.add_argument("--identity-threshold", type=float, default=0.55)
    ap.add_argument("--skip-identity", action="store_true")
    args = ap.parse_args()

    roots = dict(r.split("=", 1) for r in args.roots)

    df = scan_roots(roots)
    print(f"[1/4] scanned {len(df)} images from {len(roots)} sources")

    df = compute_hashes(df)
    print("[2/4] hashes computed")

    df = flag_duplicates(df, args.phash_threshold)
    print(f"[3/4] {df['is_exact_dup'].sum()} exact dups, "
          f"{(df['dup_cluster_size'] > 1).sum()} images in near-dup clusters")

    if args.skip_identity:
        df["group"] = df["dup_cluster"]
        df["group_source"] = "dup_cluster_only"
    else:
        df = infer_identities(df, args.identity_threshold)
    print(f"[4/4] {df['group'].nunique()} identity groups")

    df.to_csv(args.out, index=False)
    rep = audit(df)
    Path(args.audit_out).write_text(json.dumps(rep, indent=2))
    print(json.dumps(rep, indent=2))
    print(f"\nwrote {args.out} and {args.audit_out}")


if __name__ == "__main__":
    main()

# ASD Fer Baseline

In [ ]:
#!/usr/bin/env python3
"""
asd_fer_baseline.py
===================
Publication-grade baseline benchmark for ASD facial emotion recognition.

Designed for a Q1 biomedical / health-informatics venue (IEEE JBHI, Computers in
Biology and Medicine, Artificial Intelligence in Medicine, Biomedical Signal
Processing and Control). Everything a reviewer at those venues will demand is
produced by this one script.

WHAT THIS DOES THAT YOUR CURRENT SCRIPT DOES NOT
------------------------------------------------
1. SUBJECT-INDEPENDENT CV. StratifiedGroupKFold on the `group` column from
   build_manifest.py, so no child appears in both train and test.
2. A LOCKED HELD-OUT TEST SET carved out by group BEFORE any CV, touched exactly
   once at the very end. Model selection never sees it.
3. MULTI-SEED REPETITION. Every model x fold is run over N seeds so you can report
   mean +/- SD and separate real gaps from initialisation noise.
4. FULL PREDICTION LOGGING. Per-image probabilities, labels, groups and sources are
   written to .npz. Every downstream statistic is recomputed from these files, so
   your numbers are reproducible and auditable.
5. CLUSTER BOOTSTRAP CIs. Resampling is done over GROUPS, not images -- image-level
   bootstrap on clustered data gives CIs that are far too narrow.
6. SIGNIFICANCE TESTING. McNemar (paired, per-image) and Wilcoxon signed-rank (paired,
   per-fold) with Holm-Bonferroni correction across the 10-model family.
7. CALIBRATION. ECE, MCE, Brier, reliability curves, temperature scaling fitted on
   validation only -- clinical reviewers ask for this.
8. SELECTIVE PREDICTION. Risk-coverage curve + AURC, i.e. "if the model abstains on
   its least confident 20%, how good is it on the rest?" This is the clinically
   meaningful framing for a screening-adjacent tool.
9. LEAVE-ONE-DATASET-OUT + SOURCE PROBE. Quantifies how much of your accuracy is
   dataset artefact rather than emotion signal.
10. GRAD-CAM export for the interpretability figure.

USAGE
-----
    # 0. build the manifest first
    python build_manifest.py --roots nora=... ferac=... --out manifest.csv

    # 1. full benchmark
    python asd_fer_baseline.py --manifest manifest.csv --out-dir runs/baseline \
        --models vgg16 swin_base_patch4_window7_224 inception_v3 deit_small_patch16_224 \
                 densenet121 vit_base_patch16_224 swin_tiny_patch4_window7_224 \
                 efficientnet_b0 mobilenetv2_100 resnet50 \
        --seeds 0 1 2 --folds 5 --epochs 30

    # 2. analysis only (re-runs every statistic from saved predictions, seconds)
    python asd_fer_baseline.py --manifest manifest.csv --out-dir runs/baseline --analyze-only

    # 3. leave-one-dataset-out
    python asd_fer_baseline.py --manifest manifest.csv --out-dir runs/lodo --lodo \
        --models vgg16 swin_base_patch4_window7_224

    # 4. dataset-source probe (the confound check)
    python asd_fer_baseline.py --manifest manifest.csv --out-dir runs/probe --source-probe

DEPENDENCIES
------------
    pip install torch torchvision timm scikit-learn scipy pandas numpy matplotlib tqdm
"""

from __future__ import annotations

import argparse
import copy
import json
import os
import platform
import random
import subprocess
import sys
import time
from dataclasses import dataclass, asdict, field
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm import tqdm

CLASSES = ["anger", "fear", "joy", "natural", "sadness", "surprise"]
C2I = {c: i for i, c in enumerate(CLASSES)}


# ======================================================================================
# Config & reproducibility
# ======================================================================================
@dataclass
class Config:
    manifest: str = "manifest.csv"
    out_dir: str = "runs/baseline"
    models: list = field(default_factory=lambda: ["resnet50"])
    img_size: int = 224
    batch_size: int = 32
    epochs: int = 30
    folds: int = 5
    seeds: list = field(default_factory=lambda: [0, 1, 2])
    test_frac: float = 0.15          # locked held-out fraction, split BY GROUP
    lr_head: float = 3e-4
    lr_backbone: float = 3e-5        # 10x slower, as in your current setup
    weight_decay: float = 1e-4
    warmup_epochs: int = 2
    label_smoothing: float = 0.05
    ema_decay: float = 0.999
    patience: int = 8
    num_workers: int = 4
    amp: bool = True
    drop_exact_dups: bool = True
    n_bootstrap: int = 2000


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Determinism costs ~10-20% speed. Reviewers of a reproducibility-conscious journal
    # will ask whether you enabled it. Keep it on.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)


def provenance() -> dict:
    def _git(*a):
        try:
            return subprocess.check_output(["git", *a], stderr=subprocess.DEVNULL
                                           ).decode().strip()
        except Exception:
            return None
    return {
        "python": sys.version, "platform": platform.platform(),
        "torch": torch.__version__, "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "git_commit": _git("rev-parse", "HEAD"),
        "git_dirty": bool(_git("status", "--porcelain")),
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
    }


# ======================================================================================
# Data
# ======================================================================================
class FaceDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tfm, return_meta=False):
        self.df = df.reset_index(drop=True)
        self.tfm = tfm
        self.return_meta = return_meta

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        with Image.open(r["path"]) as im:
            im = im.convert("RGB")
        x = self.tfm(im)
        y = C2I[r["label"]]
        if self.return_meta:
            return x, y, i
        return x, y


def build_transforms(size: int):
    from torchvision import transforms as T
    mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    # Deliberately conservative. Heavy augmentation (MixUp/CutMix, large affine) destroys
    # the subtle asymmetries that are the clinically interesting signal in ASD faces.
    # NOTE: horizontal flip is included by convention, but if facial ASYMMETRY is part of
    # your hypothesis, run an ablation WITHOUT flip and report it -- flipping is a
    # symmetry prior and a reviewer who reads your motivation will spot the tension.
    train = T.Compose([
        T.Resize((size + 32, size + 32)),
        T.RandomResizedCrop(size, scale=(0.80, 1.0), ratio=(0.9, 1.11)),
        T.RandomHorizontalFlip(0.5),
        T.RandomApply([T.RandomRotation(10)], p=0.5),
        T.ColorJitter(0.20, 0.20, 0.20, 0.02),
        T.ToTensor(),
        T.Normalize(mean, std),
        T.RandomErasing(p=0.25, scale=(0.02, 0.10)),
    ])
    eval_ = T.Compose([
        T.Resize((size, size)),
        T.ToTensor(),
        T.Normalize(mean, std),
    ])
    return train, eval_


def load_manifest(cfg: Config) -> pd.DataFrame:
    df = pd.read_csv(cfg.manifest)
    required = {"path", "label", "source", "group"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"manifest missing columns: {missing}. Run build_manifest.py.")
    df = df[df["label"].isin(CLASSES)].copy()
    if cfg.drop_exact_dups and "is_exact_dup" in df.columns:
        n0 = len(df)
        df = df[~df["is_exact_dup"].astype(bool)]
        print(f"[data] dropped {n0 - len(df)} exact duplicates")
    if "dup_cluster" in df.columns:
        # Keep one representative per near-duplicate cluster. Duplicates inflate both
        # training (the model memorises) and testing (the same face is scored twice).
        n0 = len(df)
        df = df.sort_values("path").drop_duplicates("dup_cluster", keep="first")
        print(f"[data] collapsed {n0 - len(df)} near-duplicates")
    return df.reset_index(drop=True)


def group_holdout(df: pd.DataFrame, frac: float, seed: int = 12345):
    """Carve a locked test set BY GROUP, keeping class proportions as close as possible.

    Greedy: shuffle groups, add until the target size is reached. Simple, deterministic,
    and reportable. The returned test set is written to disk once and NEVER regenerated.
    """
    rng = np.random.RandomState(seed)
    groups = df.groupby("group").agg(n=("path", "size"), lab=("label", "first"))
    chosen = []
    for lab in CLASSES:  # per class, so rare classes are represented in the test set
        pool = groups[groups["lab"] == lab].index.to_numpy()
        rng.shuffle(pool)
        want = max(1, int(round(len(df[df["label"] == lab]) * frac)))
        got = 0
        for g in pool:
            if got >= want:
                break
            chosen.append(g); got += groups.loc[g, "n"]
    test = df[df["group"].isin(chosen)].copy()
    dev = df[~df["group"].isin(chosen)].copy()
    return dev.reset_index(drop=True), test.reset_index(drop=True)


# ======================================================================================
# Model
# ======================================================================================
def build_model(name: str, n_classes: int = 6, pretrained: bool = True):
    import timm
    m = timm.create_model(name, pretrained=pretrained, num_classes=n_classes)
    return m


def param_groups(model, cfg: Config):
    head_names = []
    try:
        import timm
        clf = model.get_classifier()
        head_names = [n for n, p in model.named_parameters()
                      if any(n.startswith(hn) for hn in
                             [k for k, _ in model.named_modules() if _ is clf])]
    except Exception:
        pass
    if not head_names:  # fall back to name heuristics
        head_names = [n for n, _ in model.named_parameters()
                      if any(t in n for t in ("head", "fc", "classifier"))]
    head = [p for n, p in model.named_parameters() if n in head_names]
    back = [p for n, p in model.named_parameters() if n not in head_names]
    return [{"params": back, "lr": cfg.lr_backbone},
            {"params": head, "lr": cfg.lr_head}]


class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = copy.deepcopy(model).eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for s, m in zip(self.shadow.state_dict().values(),
                        model.state_dict().values()):
            if s.dtype.is_floating_point:
                s.mul_(self.decay).add_(m, alpha=1 - self.decay)
            else:
                s.copy_(m)


# ======================================================================================
# Train / evaluate one fold
# ======================================================================================
def make_sampler(labels: np.ndarray) -> WeightedRandomSampler:
    counts = np.bincount(labels, minlength=len(CLASSES)).astype(float)
    w = 1.0 / np.maximum(counts, 1)
    sw = w[labels]
    return WeightedRandomSampler(torch.as_tensor(sw, dtype=torch.double),
                                 num_samples=len(labels), replacement=True)


@torch.no_grad()
def predict(model, loader, device) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    logits, ys = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        with torch.autocast("cuda", enabled=torch.cuda.is_available()):
            out = model(x)
        if isinstance(out, (tuple, list)):   # inception_v3 aux
            out = out[0]
        logits.append(out.float().cpu().numpy())
        ys.append(y.numpy())
    return np.concatenate(logits), np.concatenate(ys)


def eval_loss(logits: np.ndarray, y: np.ndarray, label_smoothing=0.0) -> float:
    """Validation loss on the SAME definition as the training criterion, so the two
    curves in the learning-curve figure are directly comparable. A train/val loss plot
    where the two lines were computed with different losses is meaningless."""
    lg = torch.tensor(logits, dtype=torch.float32)
    yy = torch.tensor(y, dtype=torch.long)
    return float(F.cross_entropy(lg, yy, label_smoothing=label_smoothing).item())


def train_one(model_name, tr_df, va_df, cfg: Config, seed: int, device):
    from sklearn.metrics import f1_score
    seed_everything(seed)
    tr_tfm, ev_tfm = build_transforms(cfg.img_size)

    tr_ds, va_ds = FaceDataset(tr_df, tr_tfm), FaceDataset(va_df, ev_tfm)
    y_tr = np.array([C2I[l] for l in tr_df["label"]])
    tr_ld = DataLoader(tr_ds, batch_size=cfg.batch_size, sampler=make_sampler(y_tr),
                       num_workers=cfg.num_workers, pin_memory=True, drop_last=True)
    va_ld = DataLoader(va_ds, batch_size=cfg.batch_size * 2, shuffle=False,
                       num_workers=cfg.num_workers, pin_memory=True)

    model = build_model(model_name).to(device)
    opt = torch.optim.AdamW(param_groups(model, cfg), weight_decay=cfg.weight_decay)
    steps = max(1, len(tr_ld))
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=[cfg.lr_backbone, cfg.lr_head],
        total_steps=cfg.epochs * steps,
        pct_start=cfg.warmup_epochs / max(cfg.epochs, 1), anneal_strategy="cos")
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.amp and torch.cuda.is_available())
    # Loss is UNWEIGHTED because the sampler already rebalances. Weighting both is
    # double-counting and is a common reviewer catch.
    crit = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    ema = EMA(model, cfg.ema_decay)

    best_f1, best_state, bad, history = -1.0, None, 0, []
    for ep in range(cfg.epochs):
        model.train()
        tot, seen, tr_pred, tr_true = 0.0, 0, [], []
        for x, y in tr_ld:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=scaler.is_enabled()):
                out = model(x)
                if isinstance(out, (tuple, list)):
                    loss = crit(out[0], y) + 0.4 * crit(out[1], y)
                    out = out[0]
                else:
                    loss = crit(out, y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt); scaler.update(); sched.step()
            ema.update(model)
            tot += loss.item() * x.size(0); seen += x.size(0)
            # running train predictions, free: reuse the forward pass already done.
            # These are on AUGMENTED, sampler-rebalanced batches, so train macro-F1 is a
            # slight under-estimate of clean train performance -- state that in the
            # caption rather than paying for a second clean pass over the training set.
            tr_pred.append(out.detach().argmax(1).cpu().numpy())
            tr_true.append(y.detach().cpu().numpy())

        lg, yy = predict(ema.shadow, va_ld, device)
        f1 = f1_score(yy, lg.argmax(1), average="macro", zero_division=0)
        tr_pred = np.concatenate(tr_pred); tr_true = np.concatenate(tr_true)
        history.append({
            "epoch": ep,
            "train_loss": tot / max(seen, 1),
            "val_loss": eval_loss(lg, yy, cfg.label_smoothing),
            "train_macro_f1": float(f1_score(tr_true, tr_pred, average="macro",
                                             zero_division=0)),
            "val_macro_f1": float(f1),
            "lr_head": float(opt.param_groups[-1]["lr"]),
        })
        if f1 > best_f1:
            best_f1, bad = f1, 0
            best_state = copy.deepcopy(ema.shadow.state_dict())
        else:
            bad += 1
            if bad >= cfg.patience:
                break

    ema.shadow.load_state_dict(best_state)
    return ema.shadow, history, best_f1


# ======================================================================================
# Metrics
# ======================================================================================
def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def core_metrics(y, p):
    from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                                 precision_score, recall_score, matthews_corrcoef,
                                 cohen_kappa_score)
    yh = p.argmax(1)
    out = {
        "accuracy": accuracy_score(y, yh),
        "balanced_accuracy": balanced_accuracy_score(y, yh),
        "f1_macro": f1_score(y, yh, average="macro", zero_division=0),
        "f1_weighted": f1_score(y, yh, average="weighted", zero_division=0),
        "precision_macro": precision_score(y, yh, average="macro", zero_division=0),
        "recall_macro": recall_score(y, yh, average="macro", zero_division=0),
        "mcc": matthews_corrcoef(y, yh),
        "kappa": cohen_kappa_score(y, yh),
    }
    try:
        import warnings
        from sklearn.metrics import roc_auc_score, average_precision_score
        Y = np.eye(len(CLASSES))[y]
        present = Y.sum(0) > 0          # bootstrap replicates can drop a rare class
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            out["auroc_ovr_macro"] = roc_auc_score(
                Y[:, present], p[:, present], average="macro", multi_class="ovr")
            out["auprc_macro"] = average_precision_score(
                Y[:, present], p[:, present], average="macro")
    except Exception:
        out["auroc_ovr_macro"] = np.nan
        out["auprc_macro"] = np.nan
    return out


def per_class_metrics(y, p):
    from sklearn.metrics import precision_recall_fscore_support
    yh = p.argmax(1)
    pr, rc, f1, sup = precision_recall_fscore_support(
        y, yh, labels=range(len(CLASSES)), zero_division=0)
    return pd.DataFrame({"class": CLASSES, "precision": pr, "recall": rc,
                         "f1": f1, "support": sup})


def expected_calibration_error(y, p, n_bins=15):
    conf = p.max(1); correct = (p.argmax(1) == y).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = mce = 0.0
    rows = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() == 0:
            rows.append((lo, hi, 0, np.nan, np.nan)); continue
        acc, avg = correct[m].mean(), conf[m].mean()
        gap = abs(acc - avg)
        ece += m.mean() * gap
        mce = max(mce, gap)
        rows.append((lo, hi, int(m.sum()), float(acc), float(avg)))
    brier = float(np.mean(np.sum((p - np.eye(len(CLASSES))[y]) ** 2, axis=1)))
    rel = pd.DataFrame(rows, columns=["lo", "hi", "n", "accuracy", "confidence"])
    return {"ece": float(ece), "mce": float(mce), "brier": brier}, rel


def temperature_scale(logits_val, y_val):
    """Fit a single temperature on VALIDATION logits. Never fit on test."""
    t = torch.nn.Parameter(torch.ones(1) * 1.0)
    lg = torch.tensor(logits_val, dtype=torch.float32)
    yy = torch.tensor(y_val, dtype=torch.long)
    opt = torch.optim.LBFGS([t], lr=0.05, max_iter=100)

    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(lg / t.clamp(min=1e-3), yy)
        loss.backward()
        return loss
    opt.step(closure)
    return float(t.detach().clamp(min=1e-3).item())


def risk_coverage(y, p):
    """Selective prediction. Sort by confidence, sweep the abstention threshold."""
    conf = p.max(1); err = (p.argmax(1) != y).astype(float)
    order = np.argsort(-conf)
    err = err[order]
    cov = np.arange(1, len(err) + 1) / len(err)
    risk = np.cumsum(err) / np.arange(1, len(err) + 1)
    aurc = float(np.trapezoid(risk, cov)) if hasattr(np, "trapezoid") \
        else float(np.trapz(risk, cov))
    return pd.DataFrame({"coverage": cov, "risk": risk}), aurc


def cluster_bootstrap_ci(y, p_list, groups, fn, n_boot=2000, alpha=0.05, seed=0):
    """Bootstrap by RESAMPLING GROUPS, not images.

    With multiple frames per child, image-level bootstrap treats correlated frames as
    independent and produces CIs roughly sqrt(frames_per_child) times too narrow. This
    is the single most common statistical error in small-face-dataset papers.

    `p_list` may be one probability matrix or a LIST of them (one per seed). With a
    list, every replicate averages the metric across seeds, so the interval is centred
    on exactly the same quantity the point estimate in the table reports. Mixing the
    two -- a mean-over-seeds point estimate next to a CI computed on the seed-ensemble
    -- produces the tell-tale "estimate outside its own interval" that a careful
    reviewer will notice immediately.
    """
    return cluster_bootstrap_multi(y, p_list, groups, {"m": fn},
                                   n_boot, alpha, seed)["m"]


def cluster_bootstrap_multi(y, p_list, groups, fns: dict, n_boot=2000,
                            alpha=0.05, seed=0):
    """Same group-level resampling, but every metric in `fns` is evaluated on ONE set
    of replicates. Bootstrapping each metric separately costs k times as much for no
    statistical benefit."""
    if isinstance(p_list, np.ndarray):
        p_list = [p_list]
    rng = np.random.RandomState(seed)
    uniq = np.unique(groups)
    idx_by_g = {g: np.where(groups == g)[0] for g in uniq}

    def point(idx):
        out = {}
        for name, fn in fns.items():
            vals = []
            for p in p_list:
                try:
                    vals.append(fn(y[idx], p[idx]))
                except Exception:
                    pass
            out[name] = float(np.mean(vals)) if vals else np.nan
        return out

    stats = {k: [] for k in fns}
    for _ in range(n_boot):
        gs = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([idx_by_g[g] for g in gs])
        if len(np.unique(y[idx])) < 2:
            continue
        for k, v in point(idx).items():
            if not np.isnan(v):
                stats[k].append(v)

    est = point(np.arange(len(y)))
    res = {}
    for k in fns:
        arr = np.asarray(stats[k], dtype=float)
        if arr.size == 0:
            res[k] = (est[k], np.nan, np.nan); continue
        lo, hi = np.percentile(arr, [100 * alpha / 2, 100 * (1 - alpha / 2)])
        # Percentile intervals on a bounded, skewed statistic can sit marginally off
        # the point estimate; clamp so the interval always contains it.
        res[k] = (float(est[k]), float(min(lo, est[k])), float(max(hi, est[k])))
    return res


def mcnemar_test(y, pa, pb):
    """Exact paired test between two models on the SAME images."""
    from scipy.stats import binomtest
    a = (pa.argmax(1) == y); b = (pb.argmax(1) == y)
    n01 = int(np.sum(a & ~b))   # A right, B wrong
    n10 = int(np.sum(~a & b))
    if n01 + n10 == 0:
        return {"n01": 0, "n10": 0, "p": 1.0, "odds": np.nan}
    p = binomtest(n01, n01 + n10, 0.5).pvalue
    return {"n01": n01, "n10": n10, "p": float(p),
            "odds": float(n01 / n10) if n10 else np.inf}


def holm(pvals: dict) -> dict:
    items = sorted(pvals.items(), key=lambda kv: kv[1])
    m = len(items)
    out, prev = {}, 0.0
    for i, (k, p) in enumerate(items):
        adj = min(1.0, max(prev, (m - i) * p))
        out[k] = adj
        prev = adj
    return out


# ======================================================================================
# Benchmark driver
# ======================================================================================
def run_benchmark(cfg: Config):
    from sklearn.model_selection import StratifiedGroupKFold

    device = "cuda" if torch.cuda.is_available() else "cpu"
    out = Path(cfg.out_dir); (out / "preds").mkdir(parents=True, exist_ok=True)
    (out / "ckpt").mkdir(exist_ok=True)
    (out / "config.json").write_text(json.dumps(
        {"config": asdict(cfg), "provenance": provenance()}, indent=2, default=str))

    df = load_manifest(cfg)

    # ---- locked test set, split by group, generated once and cached -------------------
    test_path = out / "locked_test_groups.json"
    if test_path.exists():
        keep = set(json.loads(test_path.read_text()))
        test_df = df[df["group"].isin(keep)].reset_index(drop=True)
        dev_df = df[~df["group"].isin(keep)].reset_index(drop=True)
        print("[split] reusing cached locked test set")
    else:
        dev_df, test_df = group_holdout(df, cfg.test_frac)
        test_path.write_text(json.dumps(sorted(test_df["group"].unique().tolist())))
    print(f"[split] dev={len(dev_df)} imgs / {dev_df['group'].nunique()} groups | "
          f"locked test={len(test_df)} imgs / {test_df['group'].nunique()} groups")
    assert set(dev_df["group"]) & set(test_df["group"]) == set(), "GROUP LEAK"

    y_dev = np.array([C2I[l] for l in dev_df["label"]])
    g_dev = dev_df["group"].to_numpy()

    _, ev_tfm = build_transforms(cfg.img_size)
    test_ld = DataLoader(FaceDataset(test_df, ev_tfm), batch_size=cfg.batch_size * 2,
                         shuffle=False, num_workers=cfg.num_workers)

    for model_name in cfg.models:
        for seed in cfg.seeds:
            tag = f"{model_name}__seed{seed}"
            fpath = out / "preds" / f"{tag}.npz"
            if fpath.exists():
                print(f"[skip] {tag} (resumable)"); continue

            sgkf = StratifiedGroupKFold(n_splits=cfg.folds, shuffle=True,
                                        random_state=seed)
            oof_logits = np.zeros((len(dev_df), len(CLASSES)), dtype=np.float32)
            oof_fold = np.full(len(dev_df), -1, dtype=np.int64)
            test_logits_folds, fold_hist = [], []

            for k, (tr_i, va_i) in enumerate(sgkf.split(dev_df, y_dev, groups=g_dev)):
                assert not (set(g_dev[tr_i]) & set(g_dev[va_i])), "fold GROUP LEAK"
                t0 = time.time()
                model, hist, best = train_one(model_name, dev_df.iloc[tr_i],
                                              dev_df.iloc[va_i], cfg, seed + 100 * k,
                                              device)
                va_ld = DataLoader(FaceDataset(dev_df.iloc[va_i], ev_tfm),
                                   batch_size=cfg.batch_size * 2, shuffle=False,
                                   num_workers=cfg.num_workers)
                lg, _ = predict(model, va_ld, device)
                oof_logits[va_i] = lg
                oof_fold[va_i] = k
                tl, _ = predict(model, test_ld, device)
                test_logits_folds.append(tl)
                fold_hist.append({"fold": k, "best_val_macro_f1": best,
                                  "minutes": (time.time() - t0) / 60,
                                  "history": hist})
                print(f"[{tag}] fold {k}: val macroF1={best:.4f} "
                      f"({(time.time()-t0)/60:.1f} min)")
                del model; torch.cuda.empty_cache()

            np.savez_compressed(
                fpath,
                oof_logits=oof_logits, oof_fold=oof_fold,
                y_dev=y_dev, group_dev=g_dev, source_dev=dev_df["source"].to_numpy(),
                path_dev=dev_df["path"].to_numpy(),
                test_logits=np.stack(test_logits_folds),
                y_test=np.array([C2I[l] for l in test_df["label"]]),
                group_test=test_df["group"].to_numpy(),
                source_test=test_df["source"].to_numpy(),
                path_test=test_df["path"].to_numpy(),
            )
            (out / "preds" / f"{tag}_history.json").write_text(
                json.dumps(fold_hist, indent=2))
            print(f"[saved] {fpath}")


# ======================================================================================
# Analysis (runs from saved .npz -- fast, deterministic, reproducible)
# ======================================================================================
def analyze(cfg: Config):
    out = Path(cfg.out_dir)
    files = sorted((out / "preds").glob("*.npz"))
    if not files:
        sys.exit(f"no predictions in {out/'preds'}")
    (out / "tables").mkdir(exist_ok=True)
    (out / "figures").mkdir(exist_ok=True)

    runs = {}
    for f in files:
        model, seed = f.stem.split("__seed")
        runs.setdefault(model, {})[int(seed)] = np.load(f, allow_pickle=True)

    # ---------- Table 2: OOF benchmark, mean +/- SD over seeds, with cluster CI --------
    rows, oof_pool, test_pool, oof_seeds = [], {}, {}, {}
    for model, byseed in runs.items():
        seeds_sorted = sorted(byseed)
        p_seeds = [softmax(byseed[s]["oof_logits"]) for s in seeds_sorted]
        per_seed = [core_metrics(byseed[s]["y_dev"], p)
                    for s, p in zip(seeds_sorted, p_seeds)]
        agg = {k: (np.mean([m[k] for m in per_seed]),
                   np.std([m[k] for m in per_seed], ddof=1) if len(per_seed) > 1 else 0.0)
               for k in per_seed[0]}

        d0 = byseed[seeds_sorted[0]]
        oof_seeds[model] = (d0["y_dev"], p_seeds, d0["group_dev"])
        # seed-ensembled probabilities: the object all PAIRED tests operate on, so that
        # McNemar compares one deterministic prediction per image per model
        p_mean = np.mean(p_seeds, axis=0)
        oof_pool[model] = (d0["y_dev"], p_mean, d0["group_dev"])
        pt_seeds = [softmax(byseed[s]["test_logits"].mean(0)) for s in seeds_sorted]
        test_pool[model] = (d0["y_test"], np.mean(pt_seeds, axis=0), d0["group_test"])

        row = {"model": model, "n_seeds": len(byseed)}
        for k, (m, sd) in agg.items():
            row[k] = m
            row[k + "_sd"] = sd
        # CI is bootstrapped over the SAME per-seed mean the point estimate reports.
        # Cheap closed-form metrics only -- AUROC inside 2000 replicates is needless.
        from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                                     f1_score as _f1)
        ci = cluster_bootstrap_multi(
            d0["y_dev"], p_seeds, d0["group_dev"],
            {"accuracy": lambda a, b: accuracy_score(a, b.argmax(1)),
             "balanced_accuracy": lambda a, b: balanced_accuracy_score(a, b.argmax(1)),
             "f1_macro": lambda a, b: _f1(a, b.argmax(1), average="macro",
                                          zero_division=0)},
            n_boot=cfg.n_bootstrap)
        for k, (v, lo, hi) in ci.items():
            row[k] = v          # keep table and interval on one definition
            row[f"{k}_ci_lo"], row[f"{k}_ci_hi"] = lo, hi
        rows.append(row)

    bench = pd.DataFrame(rows).sort_values("f1_macro", ascending=False)
    bench.to_csv(out / "tables" / "T2_benchmark_oof.csv", index=False)

    # publication-ready string column: 0.721 [0.683, 0.759]
    def fmt(r, k):
        return f"{r[k]:.3f} [{r[k+'_ci_lo']:.3f}, {r[k+'_ci_hi']:.3f}]"
    pretty = pd.DataFrame({
        "Model": bench["model"],
        "Accuracy [95% CI]": bench.apply(lambda r: fmt(r, "accuracy"), axis=1),
        "Balanced Acc [95% CI]": bench.apply(lambda r: fmt(r, "balanced_accuracy"), axis=1),
        "Macro-F1 [95% CI]": bench.apply(lambda r: fmt(r, "f1_macro"), axis=1),
        "MCC": bench["mcc"].map("{:.3f}".format),
        "AUROC": bench["auroc_ovr_macro"].map("{:.3f}".format),
        "Seed SD (F1)": bench["f1_macro_sd"].map("{:.3f}".format),
    })
    pretty.to_csv(out / "tables" / "T2_benchmark_pretty.csv", index=False)
    print("\n=== Table 2: OOF benchmark ===")
    print(pretty.to_string(index=False))

    # ---------- Table 3: paired significance vs the top model -------------------------
    best = bench.iloc[0]["model"]
    yb, pb, _ = oof_pool[best]
    tests = {m: mcnemar_test(yb, pb, p) for m, (_, p, _) in oof_pool.items()
             if m != best}
    raw = {m: t["p"] for m, t in tests.items()}
    adj = holm(raw)
    sig = pd.DataFrame([
        {"reference": best, "model": m,
         "n01_ref_right": t["n01"], "n10_ref_wrong": t["n10"],
         "mcnemar_p": t["p"], "holm_adj_p": adj[m],
         "significant_0.05": adj[m] < 0.05}
        for m, t in tests.items()
    ]).sort_values("holm_adj_p")
    sig.to_csv(out / "tables" / "T3_significance.csv", index=False)
    print(f"\n=== Table 3: McNemar vs {best} (Holm-corrected) ===")
    print(sig.to_string(index=False))
    n_sig = int(sig["significant_0.05"].sum())
    print(f"\n>>> {n_sig}/{len(sig)} models differ significantly from the best model.")
    if n_sig == 0:
        print(">>> READ THIS: no baseline is significantly better than any other. Do NOT "
              "write 'VGG-16 is the strongest architecture'. Write that the ten "
              "backbones are statistically indistinguishable at this sample size, and "
              "that this is itself a finding that motivates your hybrid design.")

    # ---------- Table 4: per-class, with the sample sizes that limit them --------------
    pc = per_class_metrics(*oof_pool[best][:2])
    pc.insert(0, "model", best)
    pc.to_csv(out / "tables" / "T4_per_class.csv", index=False)
    print("\n=== Table 4: per-class (best model) ===")
    print(pc.to_string(index=False))
    thin = pc[pc["support"] < 30]
    if len(thin):
        print(f">>> classes with n<30 ({', '.join(thin['class'])}): report CIs, and say "
              "plainly in the Limitations that these estimates are unstable.")

    # ---------- Table 5: calibration + selective prediction ---------------------------
    cal_rows = []
    for model, (y, p, g) in oof_pool.items():
        cal, rel = expected_calibration_error(y, p)
        rc, aurc = risk_coverage(y, p)
        # accuracy if the model abstains on its least-confident 20%
        acc80 = float(1 - rc.iloc[int(0.8 * len(rc)) - 1]["risk"])
        cal_rows.append({"model": model, **cal, "aurc": aurc,
                         "accuracy_at_80pct_coverage": acc80})
        rel.to_csv(out / "tables" / f"reliability_{model}.csv", index=False)
    cal_df = pd.DataFrame(cal_rows).sort_values("ece")
    cal_df.to_csv(out / "tables" / "T5_calibration.csv", index=False)
    print("\n=== Table 5: calibration & selective prediction ===")
    print(cal_df.to_string(index=False))

    # ---------- Table 6: subgroup / source breakdown ----------------------------------
    d0 = runs[best][min(runs[best])]
    src = d0["source_dev"]
    sub = []
    y, p, _ = oof_pool[best]
    for s in np.unique(src):
        m = src == s
        if m.sum() < 20:
            continue
        sub.append({"subgroup": f"source={s}", "n": int(m.sum()),
                    **core_metrics(y[m], p[m])})
    if sub:
        sdf = pd.DataFrame(sub)
        sdf.to_csv(out / "tables" / "T6_subgroup_by_source.csv", index=False)
        print("\n=== Table 6: performance by source dataset ===")
        print(sdf[["subgroup", "n", "accuracy", "f1_macro"]].to_string(index=False))
        spread = sdf["f1_macro"].max() - sdf["f1_macro"].min()
        if spread > 0.10:
            print(f">>> macro-F1 varies by {spread:.3f} across source datasets. That is "
                  "a dataset-shift confound; report it and run the LODO experiment.")

    # ---------- Locked test set: touch once, at the end -------------------------------
    from sklearn.metrics import f1_score as _f1s
    trows = []
    for model, (y, p, g) in test_pool.items():
        v, lo, hi = cluster_bootstrap_ci(
            y, p, g, lambda a, b: _f1s(a, b.argmax(1), average="macro", zero_division=0),
            n_boot=cfg.n_bootstrap)
        trows.append({"model": model, **core_metrics(y, p),
                      "f1_macro_ci_lo": lo, "f1_macro_ci_hi": hi})
    tdf = pd.DataFrame(trows).sort_values("f1_macro", ascending=False)
    tdf.to_csv(out / "tables" / "T7_locked_test.csv", index=False)
    print("\n=== Table 7: LOCKED HELD-OUT TEST (report once, never tune on it) ===")
    print(tdf[["model", "accuracy", "balanced_accuracy", "f1_macro",
               "f1_macro_ci_lo", "f1_macro_ci_hi"]].to_string(index=False))

    # ---------- Figures ---------------------------------------------------------------
    try:
        from asd_fer_figures import build_all
        build_all(out, manifest=cfg.manifest if Path(cfg.manifest).exists() else None)
    except ImportError:
        print("[warn] asd_fer_figures.py not found -- writing the minimal figure set only")
        _figures(out, bench, oof_pool, best)
    print(f"\nAll tables -> {out/'tables'}    All figures -> {out/'figures'}")


def _figures(out: Path, bench, oof_pool, best):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sklearn.metrics import confusion_matrix

    # F1 with CI -- replaces the grouped bar chart, which hid the uncertainty
    fig, ax = plt.subplots(figsize=(7, 4.5))
    b = bench.sort_values("f1_macro")
    lo_err = np.clip(b["f1_macro"] - b["f1_macro_ci_lo"], 0, None)
    hi_err = np.clip(b["f1_macro_ci_hi"] - b["f1_macro"], 0, None)
    ax.errorbar(b["f1_macro"], range(len(b)), xerr=[lo_err, hi_err],
                fmt="o", capsize=3, color="#2b6cb0")
    ax.set_yticks(range(len(b))); ax.set_yticklabels(b["model"], fontsize=8)
    ax.set_xlabel("Macro-F1 (95% cluster-bootstrap CI)")
    ax.grid(axis="x", alpha=.3); fig.tight_layout()
    fig.savefig(out / "figures" / "F1_macroF1_with_CI.png", dpi=300); plt.close(fig)

    # row-normalised confusion matrix of the best model
    y, p, _ = oof_pool[best]
    cm = confusion_matrix(y, p.argmax(1), labels=range(len(CLASSES)))
    cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(5.5, 4.8))
    im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(6), CLASSES, rotation=45, ha="right")
    ax.set_yticks(range(6), CLASSES)
    for i in range(6):
        for j in range(6):
            ax.text(j, i, f"{cmn[i,j]:.2f}\n({cm[i,j]})", ha="center", va="center",
                    fontsize=7, color="white" if cmn[i, j] > .5 else "black")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"{best} (row-normalised)")
    fig.colorbar(im); fig.tight_layout()
    fig.savefig(out / "figures" / "F2_confusion_best.png", dpi=300); plt.close(fig)

    # reliability
    fig, ax = plt.subplots(figsize=(4.6, 4.4))
    for model, (yy, pp, _) in list(oof_pool.items())[:4]:
        _, rel = expected_calibration_error(yy, pp)
        r = rel.dropna()
        ax.plot(r["confidence"], r["accuracy"], "o-", ms=3, label=model, lw=1)
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
    ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy")
    ax.legend(fontsize=6); ax.grid(alpha=.3); fig.tight_layout()
    fig.savefig(out / "figures" / "F3_reliability.png", dpi=300); plt.close(fig)

    # risk-coverage
    fig, ax = plt.subplots(figsize=(4.6, 4.0))
    for model, (yy, pp, _) in list(oof_pool.items())[:4]:
        rc, aurc = risk_coverage(yy, pp)
        ax.plot(rc["coverage"], rc["risk"], lw=1.2, label=f"{model} (AURC={aurc:.3f})")
    ax.set_xlabel("Coverage"); ax.set_ylabel("Selective risk")
    ax.legend(fontsize=6); ax.grid(alpha=.3); fig.tight_layout()
    fig.savefig(out / "figures" / "F4_risk_coverage.png", dpi=300); plt.close(fig)


# ======================================================================================
# Confound experiments
# ======================================================================================
def source_probe(cfg: Config):
    """Train a classifier to predict the SOURCE DATASET instead of the emotion.

    If this reaches high accuracy, your emotion models can trivially identify the source
    too -- and since class balance differs across sources, part of your emotion accuracy
    is dataset artefact. Reviewers at JBHI/CBM ask for exactly this on merged corpora.
    Report the probe accuracy in the paper and treat a high value as a limitation you
    address via LODO evaluation.
    """
    from sklearn.model_selection import StratifiedGroupKFold
    from sklearn.metrics import accuracy_score, balanced_accuracy_score

    device = "cuda" if torch.cuda.is_available() else "cpu"
    df = load_manifest(cfg)
    src = sorted(df["source"].unique())
    s2i = {s: i for i, s in enumerate(src)}
    probe_df = df.copy()
    probe_df["label"] = probe_df["source"]

    global CLASSES, C2I
    old_c, old_i = CLASSES, C2I
    CLASSES, C2I = src, s2i
    try:
        y = np.array([s2i[s] for s in probe_df["source"]])
        g = probe_df["group"].to_numpy()
        sgkf = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=0)
        tr, va = next(iter(sgkf.split(probe_df, y, groups=g)))
        cfg2 = copy.deepcopy(cfg); cfg2.epochs = min(cfg.epochs, 10)
        model, _, _ = train_one("resnet50", probe_df.iloc[tr], probe_df.iloc[va],
                                cfg2, 0, device)
        _, ev = build_transforms(cfg.img_size)
        ld = DataLoader(FaceDataset(probe_df.iloc[va], ev), batch_size=64, shuffle=False,
                        num_workers=cfg.num_workers)
        lg, yy = predict(model, ld, device)
        acc = accuracy_score(yy, lg.argmax(1))
        bacc = balanced_accuracy_score(yy, lg.argmax(1))
    finally:
        CLASSES, C2I = old_c, old_i

    chance = 1.0 / len(src)
    res = {"n_sources": len(src), "chance": chance,
           "probe_accuracy": float(acc), "probe_balanced_accuracy": float(bacc)}
    Path(cfg.out_dir).mkdir(parents=True, exist_ok=True)
    (Path(cfg.out_dir) / "source_probe.json").write_text(json.dumps(res, indent=2))
    print(json.dumps(res, indent=2))
    if bacc > 0.75:
        print(">>> STRONG source confound. Images carry an obvious dataset fingerprint "
              "(resolution, compression, colour). Mitigate: uniform preprocessing "
              "(same face detector, same crop, same JPEG quality, same resize), and "
              "report LODO results as your primary generalisation evidence.")
    return res


def run_lodo(cfg: Config):
    """Leave-one-dataset-out: train on 3 sources, test on the 4th. The honest estimate
    of how the model behaves on a cohort it has never seen."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    df = load_manifest(cfg)
    out = Path(cfg.out_dir); out.mkdir(parents=True, exist_ok=True)
    _, ev = build_transforms(cfg.img_size)
    rows = []
    for held in sorted(df["source"].unique()):
        tr_all = df[df["source"] != held]
        te = df[df["source"] == held]
        if len(te) < 30 or te["label"].nunique() < 2:
            continue
        # inner validation split by group, for early stopping only
        tr, va = group_holdout(tr_all, 0.15, seed=7)
        for model_name in cfg.models:
            model, _, _ = train_one(model_name, tr, va, cfg, cfg.seeds[0], device)
            ld = DataLoader(FaceDataset(te, ev), batch_size=64, shuffle=False,
                            num_workers=cfg.num_workers)
            lg, yy = predict(model, ld, device)
            p = softmax(lg)
            m = core_metrics(yy, p)
            v, lo, hi = cluster_bootstrap_ci(
                yy, p, te["group"].to_numpy(),
                lambda a, b: core_metrics(a, b)["f1_macro"], n_boot=1000)
            rows.append({"held_out_source": held, "model": model_name, "n_test": len(te),
                         **m, "f1_macro_ci_lo": lo, "f1_macro_ci_hi": hi})
            print(f"[LODO] {model_name} | held-out={held}: "
                  f"macroF1={m['f1_macro']:.3f} [{lo:.3f}, {hi:.3f}]")
            del model; torch.cuda.empty_cache()
    pd.DataFrame(rows).to_csv(out / "T8_lodo.csv", index=False)
    print(f"wrote {out/'T8_lodo.csv'}")


# ======================================================================================
# Grad-CAM for the interpretability figure
# ======================================================================================
def gradcam(model, x, target_layer, class_idx=None):
    """Minimal Grad-CAM. Works for CNNs; for ViT/Swin pass the last norm layer and
    reshape, or use attention rollout instead."""
    acts, grads = {}, {}
    h1 = target_layer.register_forward_hook(lambda m, i, o: acts.setdefault("a", o))
    h2 = target_layer.register_full_backward_hook(
        lambda m, gi, go: grads.setdefault("g", go[0]))
    model.zero_grad()
    out = model(x)
    if isinstance(out, (tuple, list)):
        out = out[0]
    if class_idx is None:
        class_idx = out.argmax(1)
    out.gather(1, class_idx[:, None]).sum().backward()
    a, g = acts["a"], grads["g"]
    h1.remove(); h2.remove()
    if a.dim() == 3:                      # transformer tokens (B, N, C) -> square map
        b, n, c = a.shape
        s = int(round((n - 1) ** 0.5))
        a = a[:, 1:, :].transpose(1, 2).reshape(b, c, s, s)
        g = g[:, 1:, :].transpose(1, 2).reshape(b, c, s, s)
    w = g.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * a).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
    cam = cam.squeeze(1)
    cam = (cam - cam.amin((1, 2), True)) / (cam.amax((1, 2), True) -
                                            cam.amin((1, 2), True) + 1e-8)
    return cam.detach().cpu().numpy()


# ======================================================================================
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--manifest", default="manifest.csv")
    ap.add_argument("--out-dir", default="runs/baseline")
    ap.add_argument("--models", nargs="+", default=["resnet50"])
    ap.add_argument("--seeds", nargs="+", type=int, default=[0, 1, 2])
    ap.add_argument("--folds", type=int, default=5)
    ap.add_argument("--epochs", type=int, default=30)
    ap.add_argument("--batch-size", type=int, default=32)
    ap.add_argument("--img-size", type=int, default=224)
    ap.add_argument("--num-workers", type=int, default=4)
    ap.add_argument("--n-bootstrap", type=int, default=2000)
    ap.add_argument("--analyze-only", action="store_true")
    ap.add_argument("--source-probe", action="store_true")
    ap.add_argument("--lodo", action="store_true")
    a = ap.parse_args()

    cfg = Config(manifest=a.manifest, out_dir=a.out_dir, models=a.models,
                 seeds=a.seeds, folds=a.folds, epochs=a.epochs,
                 batch_size=a.batch_size, img_size=a.img_size,
                 num_workers=a.num_workers, n_bootstrap=a.n_bootstrap)

    if a.source_probe:
        source_probe(cfg); return
    if a.lodo:
        run_lodo(cfg); return
    if not a.analyze_only:
        run_benchmark(cfg)
    analyze(cfg)


if __name__ == "__main__":
    main()

# ASD Fer Figures

In [ ]:
#!/usr/bin/env python3
"""
asd_fer_figures.py
==================
Every figure the manuscript needs, generated from the saved predictions and training
histories that asd_fer_baseline.py writes. No GPU, no re-training.

    python asd_fer_figures.py --run-dir runs/baseline
    python asd_fer_figures.py --run-dir runs/baseline --compare-dir runs/leaked

DESIGN RULES BAKED IN (they are also journal requirements)
----------------------------------------------------------
* Six emotion classes are NEVER six overlaid hues. Per-class curves are drawn as
  small multiples, one class per panel, single series. Six overlapping colour-coded
  curves are unreadable for a colour-blind reader and illegible in greyscale print.
* Model overlays are capped at three, drawn in a CVD-validated three-hue order with
  distinct dash patterns as secondary encoding, so identity survives greyscale.
* Folds use one hue stepped light-to-dark (ordinal data gets an ordinal ramp).
* Magnitude (confusion matrices, agreement) uses one hue light-to-dark. Signed
  differences use blue-red diverging with a grey zero. Never a rainbow.
* No dual-axis panels anywhere. Loss and F1 have different units, so they get
  separate panels rather than two y-scales on one.
* Every figure is written at 300 dpi with a white ground and vector PDF alongside.

Palettes below were checked with a CVD/contrast validator, not chosen by eye.
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

CLASSES = ["anger", "fear", "joy", "natural", "sadness", "surprise"]

# --- validated palettes ---------------------------------------------------------------
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]      # all-pairs CVD-safe; 3 is the cap
DASH = [(None, None), (5, 2), (1.5, 1.6)]        # secondary encoding for greyscale
FOLD_RAMP = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281"]   # ordinal, one hue
SEQ = ["#eef4fd", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
BLUE_CMAP = LinearSegmentedColormap.from_list("seqblue", SEQ)
DIV_CMAP = LinearSegmentedColormap.from_list(
    "divbr", ["#0d366b", "#3987e5", "#cde2fb", "#f0efec", "#f6c3bf", "#e34948", "#8f1f1e"])
INK, INK2, INK3 = "#0b0b0b", "#52514e", "#8a8983"
GRID, RULE = "#e8e8e4", "#d4d4cf"
GOOD, WARN, CRIT = "#1baf7a", "#eda100", "#d9403f"


def style():
    plt.rcParams.update({
        "figure.facecolor": "white", "axes.facecolor": "white",
        "savefig.facecolor": "white", "savefig.bbox": "tight",
        "font.size": 8.5, "axes.titlesize": 9, "axes.labelsize": 8.5,
        "legend.fontsize": 7.5, "xtick.labelsize": 7.5, "ytick.labelsize": 7.5,
        "axes.edgecolor": RULE, "axes.linewidth": .8, "axes.labelcolor": INK2,
        "text.color": INK, "xtick.color": INK3, "ytick.color": INK3,
        "grid.color": GRID, "grid.linewidth": .7,
        "axes.grid": True, "axes.axisbelow": True, "legend.frameon": False,
        "lines.linewidth": 1.6, "lines.solid_capstyle": "round",
        "axes.spines.top": False, "axes.spines.right": False,
    })


def save(fig, out: Path, name: str):
    out.mkdir(parents=True, exist_ok=True)
    fig.savefig(out / f"{name}.png", dpi=300)
    fig.savefig(out / f"{name}.pdf")          # vector, for the camera-ready
    plt.close(fig)
    print(f"  wrote {name}.png / .pdf")


def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def short(name: str) -> str:
    """timm identifiers are long enough to blow out legends and tick labels.
    'swin_base_patch4_window7_224' -> 'swin_base'. Keep the full name in the CSVs."""
    import re
    s = re.sub(r"_patch\d+.*$", "", str(name))
    s = re.sub(r"_window\d+.*$", "", s)
    s = re.sub(r"_(224|256|384|100)$", "", s)
    return s


# ======================================================================================
# Loading
# ======================================================================================
def load_run(run_dir: Path):
    runs, hist = {}, {}
    for f in sorted((run_dir / "preds").glob("*.npz")):
        model, seed = f.stem.split("__seed")
        runs.setdefault(model, {})[int(seed)] = np.load(f, allow_pickle=True)
    for f in sorted((run_dir / "preds").glob("*_history.json")):
        model, seed = f.stem.replace("_history", "").split("__seed")
        hist.setdefault(model, {})[int(seed)] = json.loads(f.read_text())
    if not runs:
        raise SystemExit(f"no predictions found in {run_dir/'preds'}")
    return runs, hist


def pooled(runs):
    """model -> (y, mean-prob over seeds, groups, [per-seed probs])"""
    out = {}
    for m, byseed in runs.items():
        ss = sorted(byseed)
        ps = [softmax(byseed[s]["oof_logits"]) for s in ss]
        d0 = byseed[ss[0]]
        out[m] = (d0["y_dev"], np.mean(ps, 0), d0["group_dev"], ps)
    return out


def rank_models(pool):
    """Rank on the mean over seeds of per-seed macro-F1 -- the same definition the
    benchmark table and the forest plot use. Ranking on the seed-ENSEMBLE instead would
    quietly give a different 'best model' in different figures of the same paper."""
    from sklearn.metrics import f1_score
    def score(m):
        y, _, _, ps = pool[m]
        return np.mean([f1_score(y, p.argmax(1), average="macro", zero_division=0)
                        for p in ps])
    return sorted(pool, key=lambda m: -score(m))


# ======================================================================================
# F01 / F02  Learning curves  -- "train vs validation"
# ======================================================================================
def fig_learning_curves(hist, model, out: Path):
    """Train vs validation, loss and macro-F1, mean over folds with a min-max band.

    Loss and F1 get separate panels. Putting them on one panel with two y-axes is the
    single most common chart error in ML papers: the crossing point of the two lines is
    an artefact of the arbitrary scaling, and readers reliably over-read it.
    """
    seeds = hist.get(model)
    if not seeds:
        return
    folds = seeds[sorted(seeds)[0]]
    # legend goes where the curves are not: loss descends (upper right free),
    # F1 ascends (lower right free)
    keys = [("train_loss", "val_loss", "Cross-entropy loss", "upper right"),
            ("train_macro_f1", "val_macro_f1", "Macro-F1", "lower right")]

    n_ep = max(len(f["history"]) for f in folds)
    def stack(key):
        M = np.full((len(folds), n_ep), np.nan)
        for i, f in enumerate(folds):
            v = [h.get(key, np.nan) for h in f["history"]]
            M[i, :len(v)] = v
        return M

    fig, axes = plt.subplots(1, 2, figsize=(7.6, 3.0))
    for ax, (ktr, kva, ylab, legend_loc) in zip(axes, keys):
        Tr, Va = stack(ktr), stack(kva)
        if np.all(np.isnan(Tr)) and np.all(np.isnan(Va)):
            ax.set_visible(False); continue
        x = np.arange(1, n_ep + 1)
        for M, c, lab, dash in ((Tr, SERIES[0], "Train", DASH[0]),
                                (Va, SERIES[1], "Validation", DASH[1])):
            mu = np.nanmean(M, 0)
            lo, hi = np.nanmin(M, 0), np.nanmax(M, 0)
            ax.fill_between(x, lo, hi, color=c, alpha=.13, linewidth=0)
            ax.plot(x, mu, color=c, label=lab,
                    dashes=dash if dash[0] else (None, None))
        # early-stopping epoch: where validation macro-F1 peaked, on average
        vf = stack("val_macro_f1")
        if not np.all(np.isnan(vf)):
            be = int(np.nanargmax(np.nanmean(vf, 0))) + 1
            ax.axvline(be, color=INK3, lw=.8, ls=":", zorder=1)
            # placed OUTSIDE the axes, above the top spine: the only position that can
            # never collide with a curve or the legend, whatever the data does
            ax.annotate(f"selected epoch {be}", xy=(be, 1.0),
                        xycoords=("data", "axes fraction"), xytext=(0, 3),
                        textcoords="offset points", ha="center", va="bottom",
                        fontsize=6.5, color=INK3)
        ax.set_xlabel("Epoch"); ax.set_ylabel(ylab)
        ax.legend(loc=legend_loc)
    fig.suptitle(f"{short(model)} — training vs validation, mean over {len(folds)} folds "
                 f"(band = fold min–max)", y=1.10, fontsize=9)
    save(fig, out, "F01_learning_curves_train_vs_val")


def fig_learning_curves_per_fold(hist, model, out: Path):
    """Per-fold validation macro-F1. Folds are ordered, so they get an ordinal ramp
    rather than arbitrary categorical hues."""
    seeds = hist.get(model)
    if not seeds:
        return
    folds = seeds[sorted(seeds)[0]]
    fig, axes = plt.subplots(1, 2, figsize=(7.6, 3.0))
    for ax, key, ylab in ((axes[0], "train_loss", "Train loss"),
                          (axes[1], "val_macro_f1", "Validation macro-F1")):
        for i, f in enumerate(folds):
            v = [h.get(key, np.nan) for h in f["history"]]
            ax.plot(np.arange(1, len(v) + 1), v,
                    color=FOLD_RAMP[i % len(FOLD_RAMP)], label=f"Fold {i}", lw=1.3)
        ax.set_xlabel("Epoch"); ax.set_ylabel(ylab)
    axes[1].legend(ncol=2, loc="lower right")
    fig.suptitle(f"{short(model)} — per-fold learning curves", y=1.04, fontsize=9)
    save(fig, out, "F02_learning_curves_per_fold")


def fig_seed_stability(pool, out: Path):
    """Seed-to-seed spread per model. If this is comparable to the between-model gaps,
    your ranking is measuring initialisation, not architecture -- and this figure is
    how you show it in one glance."""
    from sklearn.metrics import f1_score
    order = rank_models(pool)
    fig, ax = plt.subplots(figsize=(6.4, max(2.6, .34 * len(order) + 1.0)))
    for i, m in enumerate(reversed(order)):
        y, _, _, ps = pool[m]
        vals = [f1_score(y, p.argmax(1), average="macro", zero_division=0) for p in ps]
        ax.plot(vals, [i] * len(vals), "o", ms=5, color=SERIES[0], alpha=.75,
                markeredgecolor="white", markeredgewidth=.8)
        ax.plot([min(vals), max(vals)], [i, i], color=SERIES[0], lw=1.2, alpha=.35,
                zorder=1)
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels([short(m) for m in reversed(order)])
    ax.set_xlabel("Macro-F1, one point per seed")
    ax.grid(axis="y", visible=False)
    ax.set_title("Seed-to-seed variation within each backbone", loc="left")
    save(fig, out, "F03_seed_stability")


# ======================================================================================
# F04-F05  Confusion matrices
# ======================================================================================
def _cm_panel(ax, cm, norm_rows=True, title="", show_counts=True, ylab=True):
    M = cm / np.maximum(cm.sum(1, keepdims=True), 1) if norm_rows else cm
    vmax = 1.0 if norm_rows else cm.max()
    im = ax.imshow(M, cmap=BLUE_CMAP, vmin=0, vmax=vmax)
    ax.set_xticks(range(len(CLASSES))); ax.set_yticks(range(len(CLASSES)))
    ax.set_xticklabels(CLASSES, rotation=42, ha="right")
    ax.set_yticklabels(CLASSES if ylab else [])
    ax.grid(False)
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            v = M[i, j]
            txt = f"{v:.2f}" if norm_rows else f"{int(v)}"
            if show_counts and norm_rows:
                txt += f"\n{int(cm[i,j])}"
            ax.text(j, i, txt, ha="center", va="center", fontsize=6.4,
                    color="white" if v > vmax * .55 else INK)
    ax.set_xlabel("Predicted")
    if ylab:
        ax.set_ylabel("True")
    ax.set_title(title, loc="left", fontsize=8.5)
    return im


def fig_confusion(pool, model, out: Path):
    from sklearn.metrics import confusion_matrix
    y, p, _, _ = pool[model]
    cm = confusion_matrix(y, p.argmax(1), labels=range(len(CLASSES)))
    fig, axes = plt.subplots(1, 2, figsize=(9.0, 4.0))
    _cm_panel(axes[0], cm, True, "Row-normalised (recall), counts beneath")
    im = _cm_panel(axes[1], cm, False, "Raw counts", show_counts=False, ylab=False)
    fig.colorbar(im, ax=axes[1], fraction=.046, pad=.03)
    fig.subplots_adjust(wspace=.12)
    fig.suptitle(f"Confusion matrix — {short(model)}", y=1.01, fontsize=9.5)
    save(fig, out, "F04_confusion_matrix")


def fig_confusion_all(pool, out: Path, k=6):
    from sklearn.metrics import confusion_matrix
    order = rank_models(pool)[:k]
    ncol = 3
    nrow = int(np.ceil(len(order) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.0 * ncol, 2.9 * nrow))
    for ax, m in zip(np.ravel(axes), order):
        y, p, _, _ = pool[m]
        cm = confusion_matrix(y, p.argmax(1), labels=range(len(CLASSES)))
        M = cm / np.maximum(cm.sum(1, keepdims=True), 1)
        ax.imshow(M, cmap=BLUE_CMAP, vmin=0, vmax=1)
        ax.set_xticks(range(6)); ax.set_yticks(range(6)); ax.grid(False)
        ax.set_xticklabels([c[:3] for c in CLASSES], fontsize=6)
        ax.set_yticklabels([c[:3] for c in CLASSES], fontsize=6)
        ax.set_title(short(m), loc="left", fontsize=7.5)
    for ax in np.ravel(axes)[len(order):]:
        ax.set_visible(False)
    fig.suptitle("Row-normalised confusion matrices across backbones", y=1.0, fontsize=9.5)
    save(fig, out, "F05_confusion_grid")


def fig_confusion_delta(pool_a, pool_b, model, out: Path,
                        label_a="subject-independent", label_b="image-level"):
    """The leakage figure. Difference between two protocols on the same model.

    Signed data, so a diverging map with a neutral zero -- never a sequential ramp,
    which would hide the sign.
    """
    from sklearn.metrics import confusion_matrix
    if model not in pool_a or model not in pool_b:
        return
    def rn(pool):
        y, p, _, _ = pool[model]
        cm = confusion_matrix(y, p.argmax(1), labels=range(len(CLASSES)))
        return cm / np.maximum(cm.sum(1, keepdims=True), 1)
    D = rn(pool_a) - rn(pool_b)
    lim = max(.05, np.abs(D).max())
    fig, ax = plt.subplots(figsize=(4.6, 4.0))
    im = ax.imshow(D, cmap=DIV_CMAP, norm=TwoSlopeNorm(0, -lim, lim))
    ax.set_xticks(range(6), CLASSES, rotation=42, ha="right")
    ax.set_yticks(range(6), CLASSES); ax.grid(False)
    for i in range(6):
        for j in range(6):
            ax.text(j, i, f"{D[i,j]:+.2f}", ha="center", va="center", fontsize=6.4,
                    color="white" if abs(D[i, j]) > lim * .6 else INK)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"{short(model)}: {label_a} minus {label_b}", loc="left", fontsize=8.5)
    fig.colorbar(im, ax=ax, fraction=.046, pad=.03, label="Δ recall")
    save(fig, out, "F06_confusion_delta_protocol")


# ======================================================================================
# F07-F09  ROC and precision-recall
# ======================================================================================
def _roc_band(y_bin, score, groups, n_boot=400, seed=0, n_grid=200):
    """Group-bootstrap band for the ROC curve AND an interval for the AUC.

    The band is a real vertical-averaging bootstrap: at each FPR on a common grid, the
    2.5th and 97.5th percentiles of the resampled TPR. A shaded area under a single ROC
    curve is NOT a confidence band, and captioning it as one is a reporting error --
    which is exactly the kind of thing a statistical reviewer writes up.
    """
    from sklearn.metrics import roc_curve, roc_auc_score
    rng = np.random.RandomState(seed)
    uniq = np.unique(groups)
    idx_by_g = {g: np.where(groups == g)[0] for g in uniq}
    grid = np.linspace(0, 1, n_grid)
    tprs, aucs = [], []
    for _ in range(n_boot):
        gs = rng.choice(uniq, len(uniq), True)
        idx = np.concatenate([idx_by_g[g] for g in gs])
        if len(np.unique(y_bin[idx])) < 2:
            continue
        fpr, tpr, _ = roc_curve(y_bin[idx], score[idx])
        tprs.append(np.interp(grid, fpr, tpr))
        aucs.append(roc_auc_score(y_bin[idx], score[idx]))
    if not tprs:
        return grid, None, None, np.nan, np.nan
    T = np.vstack(tprs)
    return (grid, np.percentile(T, 2.5, 0), np.percentile(T, 97.5, 0),
            float(np.percentile(aucs, 2.5)), float(np.percentile(aucs, 97.5)))


def fig_roc_per_class(pool, model, out: Path, n_boot=400):
    """Six classes as small multiples, not six overlaid hues.

    Each panel is one class, one series, with its AUC and a group-bootstrap CI. The
    reader can actually read the rare classes, and it prints in greyscale.
    """
    from sklearn.metrics import roc_curve, roc_auc_score
    y, p, g, _ = pool[model]
    fig, axes = plt.subplots(2, 3, figsize=(7.8, 5.2), sharex=True, sharey=True)
    for k, (ax, cls) in enumerate(zip(np.ravel(axes), CLASSES)):
        yb = (y == k).astype(int)
        n_pos = int(yb.sum())
        if n_pos < 2:
            ax.set_visible(False); continue
        fpr, tpr, _ = roc_curve(yb, p[:, k])
        auc = roc_auc_score(yb, p[:, k])
        grid, blo, bhi, lo, hi = _roc_band(yb, p[:, k], g, n_boot)
        ax.plot([0, 1], [0, 1], color=INK3, lw=.9, ls=(0, (2, 2)), zorder=1)
        if blo is not None:
            ax.fill_between(grid, blo, bhi, color=SERIES[0], alpha=.18, linewidth=0,
                            zorder=2)
        ax.plot(fpr, tpr, color=SERIES[0], lw=1.7, zorder=3)
        ax.set_title(f"{cls}  (n={n_pos})", loc="left", fontsize=8.5)
        ax.text(.97, .06, f"AUC {auc:.3f}\n[{lo:.3f}, {hi:.3f}]", ha="right",
                va="bottom", transform=ax.transAxes, fontsize=7, color=INK2)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1.003)
        if k >= 3: ax.set_xlabel("False positive rate")
        if k % 3 == 0: ax.set_ylabel("True positive rate")
    fig.suptitle(f"One-vs-rest ROC by emotion — {short(model)}   "
                 f"(band = 95% group-bootstrap interval on TPR)", y=1.0, fontsize=9.5)
    save(fig, out, "F07_roc_per_class")


def fig_roc_models(pool, out: Path, k=3):
    """Macro-average ROC for the top three models only. Three is the cap at which a
    categorical palette still separates for every colour-vision type on overlapping
    curves; dash patterns carry identity in greyscale."""
    from sklearn.metrics import roc_curve, auc as _auc
    order = rank_models(pool)[:k]
    fig, axes = plt.subplots(1, 2, figsize=(7.6, 3.5))
    for i, m in enumerate(order):
        y, p, _, _ = pool[m]
        Y = np.eye(len(CLASSES))[y]
        # macro: interpolate each class onto a common grid, then average
        grid = np.linspace(0, 1, 400)
        tprs = []
        for c in range(len(CLASSES)):
            if Y[:, c].sum() < 2: continue
            fpr, tpr, _ = roc_curve(Y[:, c], p[:, c])
            tprs.append(np.interp(grid, fpr, tpr))
        mac = np.mean(tprs, 0)
        axes[0].plot(grid, mac, color=SERIES[i], lw=1.7,
                     dashes=DASH[i] if DASH[i][0] else (None, None),
                     label=f"{short(m)}  (AUC {_auc(grid, mac):.3f})")
        fpr, tpr, _ = roc_curve(Y.ravel(), p.ravel())      # micro
        axes[1].plot(fpr, tpr, color=SERIES[i], lw=1.7,
                     dashes=DASH[i] if DASH[i][0] else (None, None),
                     label=f"{short(m)}  (AUC {_auc(fpr, tpr):.3f})")
    for ax, t in zip(axes, ("Macro-average (each class weighted equally)",
                            "Micro-average (each image weighted equally)")):
        ax.plot([0, 1], [0, 1], color=INK3, lw=.9, ls=(0, (2, 2)), zorder=1)
        ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
        ax.set_title(t, loc="left", fontsize=8.5)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1.003)
        ax.legend(loc="lower right")
    save(fig, out, "F08_roc_models")


def fig_pr_per_class(pool, model, out: Path):
    """Precision-recall by class, with each class's prevalence drawn as the no-skill
    baseline. On a 10:1 imbalanced problem PR is the more honest curve, and without the
    prevalence line a reader cannot tell a good AP from a trivial one."""
    from sklearn.metrics import precision_recall_curve, average_precision_score
    y, p, _, _ = pool[model]
    fig, axes = plt.subplots(2, 3, figsize=(7.8, 5.2), sharex=True, sharey=True)
    for k, (ax, cls) in enumerate(zip(np.ravel(axes), CLASSES)):
        yb = (y == k).astype(int)
        if yb.sum() < 2:
            ax.set_visible(False); continue
        pr, rc, _ = precision_recall_curve(yb, p[:, k])
        ap = average_precision_score(yb, p[:, k])
        base = yb.mean()
        ax.axhline(base, color=INK3, lw=.9, ls=(0, (2, 2)), zorder=1)
        ax.fill_between(rc, pr, color=SERIES[0], alpha=.10, linewidth=0, step="post")
        ax.step(rc, pr, where="post", color=SERIES[0], lw=1.7)
        ax.set_title(f"{cls}  (n={int(yb.sum())})   AP {ap:.3f}", loc="left",
                     fontsize=8.5)
        # the prevalence label goes at recall~0, where the curve is pinned near 1.0 and
        # can never collide with it, whatever the class balance
        ax.text(.012, base + .022, f"chance {base:.3f}", transform=ax.get_yaxis_transform(),
                fontsize=6.6, color=INK3, va="bottom", ha="left")
        ax.set_xlim(0, 1); ax.set_ylim(0, 1.003)
        if k >= 3: ax.set_xlabel("Recall")
        if k % 3 == 0: ax.set_ylabel("Precision")
    fig.suptitle(f"Precision–recall by emotion — {short(model)} "
                 f"(dashed line = class prevalence)", y=1.0, fontsize=9.5)
    save(fig, out, "F09_pr_per_class")


# ======================================================================================
# F10-F12  Uncertainty behaviour
# ======================================================================================
def fig_calibration(pool, out: Path, k=3, n_bins=12):
    from asd_fer_baseline import expected_calibration_error
    order = rank_models(pool)[:k]
    fig, axes = plt.subplots(1, 2, figsize=(7.6, 3.4),
                             gridspec_kw={"width_ratios": [1, 1.15]})
    ax = axes[0]
    ax.plot([0, 1], [0, 1], color=INK3, lw=.9, ls=(0, (2, 2)), zorder=1,
            label="perfect calibration")
    for i, m in enumerate(order):
        y, p, _, _ = pool[m]
        cal, rel = expected_calibration_error(y, p, n_bins)
        # A bin holding a handful of images produces a wild accuracy estimate that reads
        # as a calibration failure. Plot only bins with >=10 images; ECE still uses all.
        r = rel.dropna()
        r = r[r["n"] >= 10]
        ax.plot(r["confidence"], r["accuracy"], "o-", ms=4.5, color=SERIES[i], lw=1.5,
                dashes=DASH[i] if DASH[i][0] else (None, None),
                markeredgecolor="white", markeredgewidth=.7,
                label=f"{short(m)}  (ECE {cal['ece']:.3f})")
    ax.set_xlabel("Predicted confidence"); ax.set_ylabel("Observed accuracy")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_title("Reliability  (bins with n≥10)", loc="left", fontsize=8.5)
    ax.legend(loc="lower right")

    ax = axes[1]
    y, p, _, _ = pool[order[0]]
    conf = p.max(1); ok = p.argmax(1) == y
    bins = np.linspace(0, 1, 26)
    ax.hist([conf[ok], conf[~ok]], bins=bins, stacked=True,
            color=[SERIES[2], SERIES[1]], label=["correct", "incorrect"],
            edgecolor="white", linewidth=.4)
    ax.set_xlabel("Predicted confidence"); ax.set_ylabel("Images")
    ax.set_title(f"Confidence distribution — {short(order[0])}", loc="left", fontsize=8.5)
    ax.legend(loc="upper left")
    save(fig, out, "F10_calibration")


def fig_risk_coverage(pool, out: Path, k=3):
    from asd_fer_baseline import risk_coverage
    order = rank_models(pool)[:k]
    fig, ax = plt.subplots(figsize=(5.0, 3.5))
    for i, m in enumerate(order):
        y, p, _, _ = pool[m]
        rc, aurc = risk_coverage(y, p)
        ax.plot(rc["coverage"], rc["risk"], color=SERIES[i], lw=1.7,
                dashes=DASH[i] if DASH[i][0] else (None, None),
                label=f"{short(m)}  (AURC {aurc:.3f})")
    ax.axvline(.8, color=INK3, lw=.8, ls=":")
    ax.annotate("80% coverage", (.8, ax.get_ylim()[1]), xytext=(-4, -3),
                textcoords="offset points", ha="right", va="top",
                fontsize=6.8, color=INK3)
    ax.set_xlabel("Coverage (fraction of images the model answers on)")
    ax.set_ylabel("Selective risk (error rate on those)")
    ax.set_title("Risk–coverage: what abstention buys you", loc="left", fontsize=8.5)
    ax.legend(loc="lower right")
    save(fig, out, "F11_risk_coverage")


def fig_per_class_f1(pool, model, out: Path, n_boot=600):
    """Per-class F1 with a group-bootstrap CI and the support printed on each bar.

    A per-class F1 bar chart without support and without an interval is the figure that
    lets a 14-sample class look like a finding."""
    from sklearn.metrics import f1_score
    y, p, g, _ = pool[model]
    rng = np.random.RandomState(0)
    uniq = np.unique(g); idx_by_g = {q: np.where(g == q)[0] for q in uniq}
    boots = np.full((n_boot, len(CLASSES)), np.nan)
    for b in range(n_boot):
        gs = rng.choice(uniq, len(uniq), True)
        idx = np.concatenate([idx_by_g[q] for q in gs])
        boots[b] = f1_score(y[idx], p[idx].argmax(1), labels=range(len(CLASSES)),
                            average=None, zero_division=0)
    est = f1_score(y, p.argmax(1), labels=range(len(CLASSES)), average=None,
                   zero_division=0)
    lo = np.nanpercentile(boots, 2.5, 0); hi = np.nanpercentile(boots, 97.5, 0)
    sup = np.bincount(y, minlength=len(CLASSES))

    fig, ax = plt.subplots(figsize=(6.4, 3.4))
    xs = np.arange(len(CLASSES))
    # thin bars, 2px surface gap between neighbours, rounded data end
    ax.bar(xs, est, width=.56, color=SERIES[0], edgecolor="white", linewidth=1.6,
           zorder=2)
    ax.errorbar(xs, est, yerr=[np.clip(est - lo, 0, None), np.clip(hi - est, 0, None)],
                fmt="none", ecolor=INK2, elinewidth=1.1, capsize=3, zorder=3)
    for x, v, n in zip(xs, est, sup):
        ax.text(x, .015, f"n={n}", ha="center", va="bottom", fontsize=6.6,
                color="white" if v > .18 else INK3, zorder=4)
        ax.text(x, hi[x] + .02, f"{v:.3f}", ha="center", va="bottom", fontsize=7,
                color=INK2)
    thin = sup < 30
    if thin.any():
        for x in xs[thin]:
            ax.text(x, -.085, "unstable", ha="center", fontsize=6.2, color=WARN,
                    transform=ax.get_xaxis_transform())
    ax.set_xticks(xs, CLASSES, rotation=18, ha="right")
    ax.set_ylabel("F1 (95% CI, group bootstrap)")
    ax.set_ylim(0, min(1.06, max(hi) + .12))
    ax.grid(axis="x", visible=False)
    ax.set_title(f"Per-class F1 with support — {short(model)}", loc="left", fontsize=8.5)
    save(fig, out, "F12_per_class_f1")


# ======================================================================================
# F13-F16  Comparison, agreement, errors
# ======================================================================================
def fig_forest(pool, out: Path, n_boot=800):
    from sklearn.metrics import f1_score, accuracy_score
    from asd_fer_baseline import cluster_bootstrap_multi
    order = rank_models(pool)
    rows = []
    for m in order:
        y, _, g, ps = pool[m]
        ci = cluster_bootstrap_multi(
            y, ps, g,
            {"f1": lambda a, b: f1_score(a, b.argmax(1), average="macro", zero_division=0),
             "acc": lambda a, b: accuracy_score(a, b.argmax(1))}, n_boot=n_boot)
        rows.append({"model": m, **{f"{k}_{s}": v for k, t in ci.items()
                                    for s, v in zip(("est", "lo", "hi"), t)}})
    df = pd.DataFrame(rows).sort_values("f1_est")
    fig, axes = plt.subplots(1, 2, figsize=(8.6, max(2.8, .34 * len(df) + 1.1)),
                             sharey=True)
    for ax, k, lab in ((axes[0], "f1", "Macro-F1"), (axes[1], "acc", "Accuracy")):
        ax.errorbar(df[f"{k}_est"], range(len(df)),
                    xerr=[np.clip(df[f"{k}_est"] - df[f"{k}_lo"], 0, None),
                          np.clip(df[f"{k}_hi"] - df[f"{k}_est"], 0, None)],
                    fmt="o", ms=6, color=SERIES[0], ecolor=INK2, elinewidth=1.1,
                    capsize=3, markeredgecolor="white", markeredgewidth=.9)
        ax.set_xlabel(f"{lab} (95% CI)")
        ax.grid(axis="y", visible=False)
    # the top model's interval, extended, shows which rows it overlaps
    top = df.iloc[-1]
    axes[0].axvspan(top["f1_lo"], top["f1_hi"], color=SERIES[0], alpha=.07, zorder=0)
    axes[0].set_yticks(range(len(df)), [short(m) for m in df["model"]])
    fig.suptitle("Backbone comparison — shaded band is the best model's interval; "
                 "any model overlapping it is not distinguishable from it",
                 y=1.02, fontsize=8.8)
    save(fig, out, "F13_forest_comparison")


def fig_significance_matrix(pool, out: Path):
    """Pairwise McNemar, Holm-corrected, as a matrix. Status colours, not a ramp:
    'significant' is a state, not a magnitude."""
    from asd_fer_baseline import mcnemar_test, holm
    order = rank_models(pool)
    n = len(order)
    raw = {}
    for i in range(n):
        for j in range(i + 1, n):
            a, b = order[i], order[j]
            raw[(a, b)] = mcnemar_test(pool[a][0], pool[a][1], pool[b][1])["p"]
    adj = holm(raw)
    M = np.full((n, n), np.nan)
    for (a, b), q in adj.items():
        M[order.index(a), order.index(b)] = q
        M[order.index(b), order.index(a)] = q
    fig, ax = plt.subplots(figsize=(max(4.8, .52 * n + 2.4), max(4.2, .52 * n + 1.9)))
    disp = np.where(np.isnan(M), np.nan, np.clip(M, 1e-30, 1))
    L = -np.log10(disp)
    # Cap the colour scale at p=1e-4. A single p=1e-18 pair would otherwise compress
    # every meaningful gradation near 0.05 into one indistinguishable shade.
    VMAX = 4.0
    im = ax.imshow(L, cmap=BLUE_CMAP, vmin=0, vmax=VMAX)
    ax.set_xticks(range(n), [short(m) for m in order], rotation=42, ha="right", fontsize=7)
    ax.set_yticks(range(n), [short(m) for m in order], fontsize=7); ax.grid(False)
    for i in range(n):
        for j in range(n):
            if i == j:
                ax.text(j, i, "—", ha="center", va="center", color=INK3, fontsize=7)
                continue
            q = M[i, j]
            sig = q < .05
            ax.text(j, i, f"{q:.3f}" if q >= .001 else "<.001", ha="center",
                    va="center", fontsize=6.2,
                    color="white" if L[i, j] / VMAX > .55 else INK,
                    fontweight="bold" if sig else "normal")
    fig.colorbar(im, ax=ax, fraction=.046, pad=.03, label="−log₁₀ adjusted p  (capped at 4)")
    ax.set_title("Pairwise McNemar, Holm-corrected (bold = significant at 0.05)",
                 loc="left", fontsize=8.5)
    save(fig, out, "F14_significance_matrix")


def fig_agreement(pool, out: Path):
    """Error-pattern agreement between backbones.

    This replaces a raw 'model correlation heatmap'. Two models agreeing on which
    images they get WRONG is what tells you an ensemble or hybrid has nothing to gain;
    correlation of scores does not. Cohen's kappa on the correct/incorrect indicator.
    """
    from sklearn.metrics import cohen_kappa_score
    order = rank_models(pool)
    n = len(order)
    K = np.eye(n)
    corr = {m: (pool[m][1].argmax(1) == pool[m][0]).astype(int) for m in order}
    for i in range(n):
        for j in range(i + 1, n):
            k = cohen_kappa_score(corr[order[i]], corr[order[j]])
            K[i, j] = K[j, i] = k
    fig, ax = plt.subplots(figsize=(max(4.8, .52 * n + 2.4), max(4.2, .52 * n + 1.9)))
    im = ax.imshow(K, cmap=BLUE_CMAP, vmin=0, vmax=1)
    ax.set_xticks(range(n), [short(m) for m in order], rotation=42, ha="right", fontsize=7)
    ax.set_yticks(range(n), [short(m) for m in order], fontsize=7); ax.grid(False)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f"{K[i,j]:.2f}", ha="center", va="center", fontsize=6.2,
                    color="white" if K[i, j] > .55 else INK)
    fig.colorbar(im, ax=ax, fraction=.046, pad=.03, label="Cohen's κ on correctness")
    ax.set_title("Do backbones fail on the same images?  (low κ ⇒ complementary)",
                 loc="left", fontsize=8.5)
    save(fig, out, "F15_error_agreement")


def fig_subgroup(pool, runs, model, out: Path):
    """Performance by source dataset. A wide spread here is the visual form of the
    merged-corpus confound."""
    from sklearn.metrics import f1_score, accuracy_score
    y, p, _, _ = pool[model]
    d0 = runs[model][min(runs[model])]
    src = d0["source_dev"]
    names, f1s, accs, ns = [], [], [], []
    for s in np.unique(src):
        m = src == s
        if m.sum() < 20: continue
        names.append(str(s)); ns.append(int(m.sum()))
        f1s.append(f1_score(y[m], p[m].argmax(1), average="macro", zero_division=0))
        accs.append(accuracy_score(y[m], p[m].argmax(1)))
    if not names:
        return
    fig, ax = plt.subplots(figsize=(6.2, 3.2))
    xs = np.arange(len(names))
    w = .34
    ax.bar(xs - w / 2 - .01, f1s, w, color=SERIES[0], edgecolor="white", linewidth=1.6,
           label="Macro-F1")
    ax.bar(xs + w / 2 + .01, accs, w, color=SERIES[1], edgecolor="white", linewidth=1.6,
           label="Accuracy")
    overall = f1_score(y, p.argmax(1), average="macro", zero_division=0)
    ax.axhline(overall, color=INK3, lw=.9, ls=(0, (2, 2)))
    ax.annotate(f"pooled macro-F1 {overall:.3f}", (len(names) - .5, overall),
                xytext=(0, 3), textcoords="offset points", ha="right",
                fontsize=6.8, color=INK3)
    ax.set_xticks(xs, [f"{a}\nn={b}" for a, b in zip(names, ns)])
    ax.set_ylabel("Score"); ax.set_ylim(0, 1.05)
    ax.grid(axis="x", visible=False); ax.legend(ncol=2, loc="upper right")
    ax.set_title(f"Performance by source dataset — {short(model)}", loc="left", fontsize=8.5)
    save(fig, out, "F16_subgroup_by_source")


def fig_lodo(csv_path: Path, out: Path):
    """Leave-one-dataset-out, as a model x held-out-source matrix."""
    if not csv_path.exists():
        return
    df = pd.read_csv(csv_path)
    piv = df.pivot_table(index="model", columns="held_out_source", values="f1_macro")
    fig, ax = plt.subplots(figsize=(max(4.4, .9 * piv.shape[1] + 2.6),
                                    max(2.6, .5 * piv.shape[0] + 1.7)))
    im = ax.imshow(piv.values, cmap=BLUE_CMAP, vmin=0, vmax=1)
    ax.set_xticks(range(piv.shape[1]), piv.columns, rotation=25, ha="right")
    ax.set_yticks(range(piv.shape[0]), [short(m) for m in piv.index], fontsize=7.5); ax.grid(False)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            v = piv.values[i, j]
            ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=7,
                    color="white" if v > .55 else INK)
    fig.colorbar(im, ax=ax, fraction=.046, pad=.03, label="Macro-F1 on held-out source")
    ax.set_title("Leave-one-dataset-out generalisation", loc="left", fontsize=8.5)
    save(fig, out, "F17_lodo_matrix")


def fig_dataset(manifest_csv: Path, out: Path):
    """Class x source composition. Figure 1 of the paper."""
    if not manifest_csv.exists():
        return
    df = pd.read_csv(manifest_csv)
    piv = (df.pivot_table(index="label", columns="source", values="path",
                          aggfunc="count").reindex(CLASSES).fillna(0))
    fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.3),
                             gridspec_kw={"width_ratios": [1.25, 1]})
    ax = axes[0]
    bottom = np.zeros(len(piv))
    ramp = [SEQ[2], SEQ[3], SEQ[4], SEQ[6]]
    for i, s in enumerate(piv.columns):
        ax.bar(range(len(piv)), piv[s].values, .58, bottom=bottom,
               color=ramp[i % len(ramp)], edgecolor="white", linewidth=1.6, label=s)
        bottom += piv[s].values
    for i, t in enumerate(bottom):
        ax.text(i, t + max(bottom) * .015, f"{int(t)}", ha="center", fontsize=7,
                color=INK2)
    ax.set_xticks(range(len(piv)), piv.index, rotation=18, ha="right")
    ax.set_ylabel("Images"); ax.grid(axis="x", visible=False)
    ax.set_ylim(0, max(bottom) * 1.32)          # headroom so the legend clears the bars
    ax.legend(ncol=4, fontsize=7, loc="upper center", bbox_to_anchor=(.5, 1.0))
    ax.set_title("Class composition by source", loc="left", fontsize=8.5)

    ax = axes[1]
    if "group" in df.columns:
        per = df.groupby("group").size().value_counts().sort_index()
        ax.bar(per.index, per.values, .7, color=SERIES[0], edgecolor="white",
               linewidth=1.4)
        ax.set_xlabel("Images contributed by one subject group")
        ax.set_ylabel("Number of groups"); ax.grid(axis="x", visible=False)
        ax.set_title(f"Subject groups (n={df['group'].nunique()}) — "
                     f"why image-level splits leak", loc="left", fontsize=8.5)
    else:
        ax.set_visible(False)
    save(fig, out, "F18_dataset_composition")


# ======================================================================================
def build_all(run_dir: Path, compare_dir: Path | None = None,
              manifest: Path | None = None):
    style()
    out = run_dir / "figures"
    runs, hist = load_run(run_dir)
    pool = pooled(runs)
    best = rank_models(pool)[0]
    print(f"[figures] {len(pool)} models, best = {best}")

    fig_learning_curves(hist, best, out)
    fig_learning_curves_per_fold(hist, best, out)
    fig_seed_stability(pool, out)
    fig_confusion(pool, best, out)
    fig_confusion_all(pool, out)
    if compare_dir:
        cruns, _ = load_run(Path(compare_dir))
        fig_confusion_delta(pool, pooled(cruns), best, out)
    fig_roc_per_class(pool, best, out)
    fig_roc_models(pool, out)
    fig_pr_per_class(pool, best, out)
    fig_calibration(pool, out)
    fig_risk_coverage(pool, out)
    fig_per_class_f1(pool, best, out)
    fig_forest(pool, out)
    fig_significance_matrix(pool, out)
    fig_agreement(pool, out)
    fig_subgroup(pool, runs, best, out)
    fig_lodo(run_dir / "T8_lodo.csv", out)
    if manifest:
        fig_dataset(Path(manifest), out)
    print(f"[figures] all written to {out}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--run-dir", required=True)
    ap.add_argument("--compare-dir", default=None,
                    help="a second run (e.g. image-level splits) for the delta figure")
    ap.add_argument("--manifest", default=None)
    a = ap.parse_args()
    build_all(Path(a.run_dir), a.compare_dir, a.manifest)

# Make Demo Run

In [ ]:
"""Generate a realistic synthetic run so every figure can be produced and inspected
without a GPU. Class counts, imbalance and performance level match the numbers in the
progress report; subject structure and fear/surprise confusability are simulated."""
import json, shutil
from pathlib import Path
import numpy as np, pandas as pd

CLASSES = ["anger", "fear", "joy", "natural", "sadness", "surprise"]
COUNTS = dict(anger=210, fear=87, joy=861, natural=230, sadness=459, surprise=156)
SOURCES = ["nora", "ferac", "talaat", "hasibur"]
rng = np.random.RandomState(7)

out = Path("runs/demo")
if out.exists(): shutil.rmtree(out)
(out / "preds").mkdir(parents=True)

# ---- subjects ------------------------------------------------------------------------
# ~4.2 images per child, which is what makes image-level splitting leak
rows = []
gid = 0
for cls, n in COUNTS.items():
    left = n
    while left > 0:
        k = min(left, max(1, int(rng.gamma(3.0, 1.4))))
        src = SOURCES[rng.choice(4, p=[.38, .22, .22, .18])]
        for _ in range(k):
            rows.append({"label": cls, "group": f"ID{gid:04d}", "source": src})
        gid += 1; left -= k
df = pd.DataFrame(rows).sample(frac=1, random_state=1).reset_index(drop=True)
df["path"] = [f"/data/{i:05d}.jpg" for i in range(len(df))]
df["dup_cluster"] = df["path"]
df.to_csv(out / "manifest_demo.csv", index=False)
print(f"{len(df)} images, {df['group'].nunique()} subject groups, "
      f"{len(df)/df['group'].nunique():.1f} images/subject")

y = np.array([CLASSES.index(l) for l in df["label"]])
groups = df["group"].to_numpy()
sources = df["source"].to_numpy()
N = len(df)

# hold out ~15% by group for the locked test set
ug = np.array(df["group"].unique()); rng.shuffle(ug)
test_g = set(ug[:int(.15 * len(ug))])
te = df["group"].isin(test_g).to_numpy()
dev_i, test_i = np.where(~te)[0], np.where(te)[0]

# ---- confusable class geometry -------------------------------------------------------
# fear<->surprise share AU1+AU2+AU5; anger<->sadness share brow lowering.
P = rng.randn(6, 12) * 1.0
P[CLASSES.index("surprise")] = P[CLASSES.index("fear")] * .72 + rng.randn(12) * .42
P[CLASSES.index("sadness")] = P[CLASSES.index("anger")] * .55 + rng.randn(12) * .68
subj_diff = {g: rng.randn() for g in np.unique(groups)}          # per-child difficulty
src_bias = {s: rng.randn(6) * .30 for s in SOURCES}              # dataset fingerprint

MODELS = {  # name -> (signal strength, confidence temperature)
    "vgg16": (1.32, 1.15), "swin_base_patch4_window7_224": (1.31, 1.02),
    "inception_v3": (1.30, 1.20), "deit_small_patch16_224": (1.26, 1.05),
    "densenet121": (1.25, 1.18), "vit_base_patch16_224": (1.24, 1.00),
    "swin_tiny_patch4_window7_224": (1.22, 1.04), "efficientnet_b0": (1.18, 1.22),
    "mobilenetv2_100": (1.12, 1.28), "resnet50": (1.09, 1.30),
}

def logits_for(strength, temp, seed):
    """Noise level calibrated so the simulated benchmark lands on the accuracy and
    macro-F1 actually reported in the progress document (~0.72 / ~0.62), with the same
    best-to-worst spread. Weaker backbones simply see noisier features."""
    r = np.random.RandomState(seed)
    noise = 1.55 + (1.32 - strength) * 0.60
    W = r.randn(12, 6) * .35 + P.T * 1.30
    feat = P[y] + np.array([[subj_diff[g]] for g in groups]) * .55 \
        + r.randn(N, 12) * noise
    z = feat @ W
    z += np.array([src_bias[s] for s in sources]) * .55
    z[np.arange(N), y] += r.randn(N) * .25
    return (z / temp).astype(np.float32)

for mi, (name, (s, t)) in enumerate(MODELS.items()):
    for seed in (0, 1, 2):
        # deterministic: Python's str hash is salted per process, so never seed with it
        z = logits_for(s, t, seed * 977 + mi * 13)
        oof = np.zeros((len(dev_i), 6), np.float32); oof[:] = z[dev_i]
        tl = np.stack([z[test_i] + rng.randn(len(test_i), 6) * .18 for _ in range(5)])
        np.savez_compressed(
            out / "preds" / f"{name}__seed{seed}.npz",
            oof_logits=oof, oof_fold=np.tile(np.arange(5), len(dev_i) // 5 + 1)[:len(dev_i)],
            y_dev=y[dev_i], group_dev=groups[dev_i], source_dev=sources[dev_i],
            path_dev=df["path"].to_numpy()[dev_i],
            test_logits=tl.astype(np.float32), y_test=y[test_i],
            group_test=groups[test_i], source_test=sources[test_i],
            path_test=df["path"].to_numpy()[test_i])

        # ---- training history with a genuine train/val divergence ----------------------
        hist = []
        for k in range(5):
            r = np.random.RandomState(seed * 31 + k)
            ep, H = 26, []
            for e in range(ep):
                prog = (e + 1) / ep
                trl = 1.80 * np.exp(-3.1 * prog) + .10 + r.randn() * .022
                val = 1.72 * np.exp(-3.6 * prog) + .52 + .40 * max(0, prog - .48) ** 1.7 \
                      + r.randn() * .036
                trf = min(.985, .17 + .80 * (1 - np.exp(-3.3 * prog)) + r.randn() * .014)
                vaf = min(.72, .13 + (s - .55) * .62 * (1 - np.exp(-4.1 * prog))
                          - .05 * max(0, prog - .62) + r.randn() * .019)
                H.append({"epoch": e, "train_loss": float(trl), "val_loss": float(val),
                          "train_macro_f1": float(trf), "val_macro_f1": float(vaf),
                          "lr_head": float(3e-4 * (1 + np.cos(np.pi * prog)) / 2)})
            hist.append({"fold": k, "best_val_macro_f1": max(h["val_macro_f1"] for h in H),
                         "minutes": float(9 + r.rand() * 5), "history": H})
        (out / "preds" / f"{name}__seed{seed}_history.json").write_text(json.dumps(hist))

# ---- a LODO table --------------------------------------------------------------------
lodo = []
for held in SOURCES:
    for m in list(MODELS)[:4]:
        base = MODELS[m][0]
        lodo.append({"held_out_source": held, "model": m,
                     "n_test": int((sources == held).sum()),
                     "f1_macro": float(np.clip((base - .55) * .60
                                               + rng.randn() * .045, .2, .8)),
                     "accuracy": float(np.clip((base - .5) * .62 + rng.randn() * .04,
                                               .2, .85))})
pd.DataFrame(lodo).to_csv(out / "T8_lodo.csv", index=False)
print("demo run written to", out)

# ASD_Fer_Zoo

In [ ]:
#!/usr/bin/env python3
"""
asd_fer_zoo.py
==============
Model registry and training regimes to explore on the ASD facial-emotion corpus.

THE GOVERNING FACT
------------------
You have ~1,700 development images from a small number of children. At that scale
ARCHITECTURE IS NOT THE BOTTLENECK -- your own benchmark already shows ten backbones
landing inside each other's confidence intervals. Two things move the number, and
neither of them is "try an eleventh backbone":

  1. WHAT THE WEIGHTS ALREADY KNOW.  ImageNet pretraining knows objects, not faces and
     not expressions. Initialising from a face- or expression-pretrained model, or from
     a self-supervised model with strong facial part correspondence, changes the result
     far more than swapping ResNet for Swin.
  2. HOW MUCH OF THE MODEL YOU ARE ALLOWED TO MOVE.  Full fine-tuning of an 87M-parameter
     ViT on 1,700 images is an invitation to memorise. Frozen features with a small head,
     or LoRA on the attention projections, usually wins at this scale -- and when it does,
     that IS a result worth reporting at a clinical venue.

This file gives you both axes: a tiered model registry (`MODEL_ZOO`) and four training
regimes (`build_model(..., mode=...)`) that plug into asd_fer_baseline.py unchanged.

USAGE
-----
    # list what is available and how big it is
    python asd_fer_zoo.py --list
    python asd_fer_zoo.py --list --tier foundation_frozen

    # from asd_fer_baseline.py, swap the model factory:
    #   from asd_fer_zoo import build_model
    #   model = build_model(name, n_classes=6, mode="lora")

    # sanity-check that a model and mode actually construct
    python asd_fer_zoo.py --check vit_base_patch16_dinov3.lvd1689m --mode lora
"""

from __future__ import annotations

import argparse
import math
import re
from dataclasses import dataclass, field

import torch
import torch.nn as nn


# ======================================================================================
# The registry
# ======================================================================================
@dataclass
class Entry:
    name: str                 # timm identifier, or a marker for an external checkpoint
    tier: str
    why: str
    img_size: int = 224
    source: str = "timm"      # timm | external | features
    note: str = ""


MODEL_ZOO: list[Entry] = [

    # ---------------------------------------------------------------------------------
    # TIER 1 -- CNN/Transformer hybrids you MUST include as baselines
    #
    # Your stated plan is to design a hybrid CNN-Transformer. The first thing a reviewer
    # will ask is why an off-the-shelf hybrid was not enough. If CoAtNet, MaxViT and
    # CAFormer are not in your table, that question ends the review. These are not
    # "extra backbones", they are the control condition for your contribution.
    # ---------------------------------------------------------------------------------
    Entry("coatnet_rmlp_2_rw_224.sw_in12k_ft_in1k", "hybrid_control",
          "Conv stages then relative-attention stages. The canonical CNN-Transformer "
          "hybrid; the direct competitor to whatever you build."),
    Entry("maxvit_tiny_tf_224.in1k", "hybrid_control",
          "Block-local plus grid-global attention interleaved with MBConv. A different "
          "hybridisation strategy from CoAtNet, so it tests the idea rather than one design."),
    Entry("caformer_s36.sail_in22k_ft_in1k", "hybrid_control",
          "MetaFormer: convolutional token mixing in early stages, self-attention late. "
          "Strong at small scale and cheap to train."),

    # ---------------------------------------------------------------------------------
    # TIER 2 -- Self-supervised foundation features, used FROZEN
    #
    # At n=1,700 a frozen encoder with a small trained head is a serious contender, not a
    # weak baseline. DINOv3 in particular carries region-level facial correspondence
    # without any face-specific training, and its INTERMEDIATE blocks carry it better
    # than its final block -- which is why `probe_layers` below pulls from mid-depth.
    # ---------------------------------------------------------------------------------
    Entry("vit_base_patch16_dinov3.lvd1689m", "foundation_frozen",
          "Best facial part correspondence of any general-purpose encoder. Use frozen, "
          "probing intermediate blocks. This is the highest expected-value single "
          "experiment on the list.", img_size=224),
    Entry("vit_large_patch16_dinov3.lvd1689m", "foundation_frozen",
          "Larger DINOv3. Frozen only -- do not attempt to fine-tune 300M parameters on "
          "1,700 images.", img_size=224),
    Entry("convnext_base.dinov3_lvd1689m", "foundation_frozen",
          "DINOv3 distilled into a ConvNeXt. Cheaper, and gives you a convolutional "
          "counterpart under identical pretraining -- a clean architectural ablation."),
    Entry("vit_base_patch14_dinov2.lvd142m", "foundation_frozen",
          "DINOv2. Include as the previous generation so the comparison is honest.",
          img_size=518),
    Entry("eva02_base_patch14_224.mim_in22k", "foundation_frozen",
          "Masked image modelling with CLIP-derived targets. Different SSL objective "
          "from DINO, so it probes whether the objective or the scale is what helps."),
    Entry("convnextv2_base.fcmae", "foundation_frozen",
          "Fully-convolutional MAE. The convolutional MIM comparison point."),
    Entry("vit_base_patch16_siglip_224.webli", "foundation_frozen",
          "Language-supervised features. Expect these to UNDERPERFORM on facial anatomy "
          "-- CLIP-style encoders localise faces but discriminate facial parts poorly. "
          "Worth one run precisely because it is the negative control."),

    # ---------------------------------------------------------------------------------
    # TIER 3 -- Expression-specialised models and FER-domain pretraining
    #
    # Not on timm; you fetch checkpoints from the authors. This is where the biggest
    # single gain probably lives, because the weights already encode expression.
    # ---------------------------------------------------------------------------------
    Entry("POSTER++", "fer_specialist", source="external",
          why="Cross-fusion of landmark and image streams; the reference FER architecture "
              "of the last few years, published in Pattern Recognition.",
          note="github.com/talented-q/poster_v2 -- RAF-DB and AffectNet checkpoints"),
    Entry("DAN", "fer_specialist", source="external",
          why="Distract-your-Attention Network. In a 2026 cross-scenario robustness "
              "study, DAN trained on AffectNet gave the strongest generalisation of any "
              "system tested, specialised or general-purpose.",
          note="github.com/yaoing/DAN -- use the AffectNet checkpoint, not RAF-DB"),
    Entry("APViT", "fer_specialist", source="external",
          why="Attentive-pooling ViT for FER; patch selection suits small datasets.",
          note="github.com/youqingxiaozhua/APViT"),
    Entry("EmoNet", "fer_specialist", source="external",
          why="Predicts valence/arousal as well as categories. The continuous outputs are "
              "a useful auxiliary signal when categorical labels are scarce and noisy.",
          note="github.com/face-analysis/emonet"),

    # ---------------------------------------------------------------------------------
    # TIER 4 -- Face-identity and face-SSL encoders
    #
    # Trained on faces at scale. Strong features for anything face-shaped, and the
    # embedding you already use for identity grouping in build_manifest.py.
    # ---------------------------------------------------------------------------------
    Entry("ArcFace-R100", "face_encoder", source="external",
          why="Face-recognition embedding. Encodes identity strongly -- which is exactly "
              "why it is a DOUBLE-EDGED baseline: if it does well on your data, that is "
              "evidence of subject leakage, not of expression modelling. Run it as a "
              "leakage probe.",
          note="insightface model zoo"),
    Entry("FaRL", "face_encoder", source="external",
          why="Vision-language pretraining on 20M face-text pairs. Face-domain "
              "equivalent of CLIP.",
          note="github.com/FacePerceiver/FaRL"),
    Entry("FSFM", "face_encoder", source="external",
          why="Self-supervised facial representation trained for face security tasks; "
              "generalises well off-distribution.",
          note="arXiv 2412.12032"),

    # ---------------------------------------------------------------------------------
    # TIER 5 -- Non-deep baselines. Do not skip these.
    #
    # If 30 action-unit intensities plus a gradient-boosted tree land inside the
    # confidence interval of an 87M-parameter transformer, that is the most interesting
    # result in your paper and the one a clinical readership will actually cite.
    # ---------------------------------------------------------------------------------
    Entry("openface_au+gbm", "interpretable", source="features",
          why="OpenFace 2.0 or Py-Feat action-unit intensities into LightGBM. ~30 "
              "features, interpretable, directly connected to the ASD facial "
              "phenomenology literature, trains in seconds."),
    Entry("landmarks+asymmetry+gbm", "interpretable", source="features",
          why="68 landmarks plus explicit left/right hemiface asymmetry statistics. This "
              "is your architectural hypothesis stated as a feature set -- if it works, "
              "it is direct evidence for the hemiface design before you build it."),
]

TIERS = {
    "hybrid_control":    "CNN-Transformer hybrids — the control your contribution must beat",
    "foundation_frozen": "Self-supervised foundation encoders, used frozen",
    "fer_specialist":    "Expression-pretrained models (external checkpoints)",
    "face_encoder":      "Face-identity / face-SSL encoders (external checkpoints)",
    "interpretable":     "Action-unit and landmark baselines",
}


# ======================================================================================
# LoRA — low-rank adaptation of attention projections
# ======================================================================================
class LoRALinear(nn.Module):
    """Wraps a frozen nn.Linear with a trainable rank-r update: W + (alpha/r)·B·A.

    Why this matters here: full fine-tuning moves ~87M parameters using ~1,700 images.
    LoRA at r=8 on the attention projections moves under 1% of that while keeping the
    pretrained representation intact. On small clinical datasets it routinely matches or
    beats full fine-tuning, and it makes seed-to-seed variance much smaller -- which,
    given how noisy your current comparisons are, is worth something on its own.
    """

    def __init__(self, base: nn.Linear, r: int = 8, alpha: int = 16, dropout: float = 0.05):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.r = r
        self.scaling = alpha / r
        self.lora_A = nn.Parameter(torch.zeros(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        self.drop = nn.Dropout(dropout)
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)      # starts as an exact no-op

    def forward(self, x):
        return self.base(x) + self.drop(x) @ self.lora_A.T @ self.lora_B.T * self.scaling


class LoRAConv1x1(nn.Module):
    """Same idea for 1x1 convolutions.

    Needed because several hybrids -- CoAtNet's relative-position variants among them --
    implement attention q/k/v and output projections as 1x1 Conv2d rather than Linear.
    A Linear-only LoRA silently finds nothing to adapt in those models.
    """

    def __init__(self, base: nn.Conv2d, r: int = 8, alpha: int = 16, dropout: float = 0.05):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.scaling = alpha / r
        self.lora_A = nn.Parameter(torch.zeros(r, base.in_channels, 1, 1))
        self.lora_B = nn.Parameter(torch.zeros(base.out_channels, r, 1, 1))
        self.drop = nn.Dropout(dropout)
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x):
        import torch.nn.functional as F
        delta = F.conv2d(F.conv2d(self.drop(x), self.lora_A), self.lora_B)
        return self.base(x) + delta * self.scaling


def inject_lora(model: nn.Module, r=8, alpha=16, dropout=0.05,
                targets=("qkv", "q_proj", "k_proj", "v_proj", "attn.proj",
                         "attn_block", "attn_grid")) -> int:
    """Wrap matching attention projections with a LoRA adapter. Returns how many.

    Only attention projections are targeted -- adapting the MLP blocks as well roughly
    doubles the trainable count for little gain at this data scale, and adapting the
    positional-bias MLPs (CoAtNet's `rel_pos.mlp.fc*`) adapts geometry rather than
    features, which is not what you want.
    """
    n = 0
    for mod_name, mod in list(model.named_modules()):
        for child_name, child in list(mod.named_children()):
            full = f"{mod_name}.{child_name}" if mod_name else child_name
            if "rel_pos" in full or not any(t in full for t in targets):
                continue
            if isinstance(child, nn.Linear):
                setattr(mod, child_name, LoRALinear(child, r, alpha, dropout)); n += 1
            elif isinstance(child, nn.Conv2d) and child.kernel_size == (1, 1):
                setattr(mod, child_name, LoRAConv1x1(child, r, alpha, dropout)); n += 1
    return n


# ======================================================================================
# Frozen multi-layer probe
# ======================================================================================
class MultiLayerProbe(nn.Module):
    """Frozen encoder + trainable head over features pooled from SEVERAL depths.

    The layer choice is not arbitrary. For DINOv3 ViT-L/16, region-level facial
    correspondence peaks around block 18 of 24 and DEGRADES by the final block, because
    late features are globally mixed. Probing only the last layer -- what a standard
    linear probe does -- throws away the part of the representation that knows a brow
    from a cheek. Defaults below pull from roughly 60%, 75% and 100% of depth.
    """

    def __init__(self, encoder: nn.Module, n_classes: int, feat_dims: list[int],
                 hidden: int = 512, dropout: float = 0.3):
        super().__init__()
        self.encoder = encoder
        for p in self.encoder.parameters():
            p.requires_grad_(False)
        self.encoder.eval()
        d = sum(feat_dims)
        self.head = nn.Sequential(
            nn.LayerNorm(d),
            nn.Linear(d, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, n_classes),
        )

    def train(self, mode=True):        # encoder stays in eval: no BN/dropout drift
        super().train(mode)
        self.encoder.eval()
        return self

    def forward(self, x):
        with torch.no_grad():
            feats = self.encoder(x)          # list of (B, N, C) or (B, C, H, W)
        pooled = []
        for f in feats:
            if f.dim() == 4:                 # conv feature map
                pooled.append(f.mean(dim=(2, 3)))
            elif f.dim() == 3:               # tokens: mean over patch tokens
                pooled.append(f.mean(dim=1))
            else:
                pooled.append(f)
        return self.head(torch.cat(pooled, dim=1).float())


def _probe_indices(name: str) -> tuple[list[int], bool]:
    """Choose which depths to probe, correctly for the two families.

    Plain transformers (ViT, DeiT, EVA) have many uniform blocks, so we take roughly
    60%, 75% and 100% of DEPTH -- bracketing the mid-network region where facial part
    correspondence is strongest, since the final block is globally mixed and worse at it.

    Hierarchical models (ConvNeXt, Swin, CAFormer, MaxViT) expose four STAGES with
    different channel widths; there the meaningful choice is the last few stages.

    Getting this wrong is not a small error. Asking timm for `features_only` and then
    indexing by the length of its default output truncates a 12-block ViT to its first
    three blocks -- a 22M-parameter stump probing early edge filters, which is the exact
    opposite of what you want.
    """
    import timm
    skel = timm.create_model(name, pretrained=False, num_classes=0)
    n_blocks = len(getattr(skel, "blocks", []) or [])
    del skel
    if n_blocks >= 8:
        return sorted({max(0, int(n_blocks * f) - 1)
                       for f in (0.60, 0.75, 1.00)}), True
    probe = timm.create_model(name, pretrained=False, features_only=True)
    n_stage = len(probe.feature_info.channels())
    del probe
    return list(range(max(0, n_stage - 3), n_stage)), False


# ======================================================================================
# Factory
# ======================================================================================
def build_model(name: str, n_classes: int = 6, mode: str = "full",
                pretrained: bool = True, lora_r: int = 8, drop_path: float = 0.1,
                probe_hidden: int = 512):
    """Build a model in one of four training regimes.

    mode:
      "full"     -- fine-tune everything (what you do now; the control)
      "lora"     -- freeze the backbone, train LoRA adapters + the head
      "probe"    -- freeze the backbone, train a head on pooled multi-depth features
      "linear"   -- freeze the backbone, train a single linear layer on final features

    Report all four for at least your top two backbones. The comparison is a genuine
    contribution at a small-clinical-data venue, and it costs you almost nothing because
    the frozen modes train in minutes.
    """
    import timm

    if mode in ("probe", "linear"):
        if mode == "linear":
            enc = timm.create_model(name, pretrained=pretrained, num_classes=0)
            dim = enc.num_features
            for p in enc.parameters():
                p.requires_grad_(False)
            enc.eval()

            class LinearProbe(nn.Module):
                def __init__(self):
                    super().__init__()
                    self.encoder, self.head = enc, nn.Linear(dim, n_classes)

                def train(self, m=True):
                    super().train(m); self.encoder.eval(); return self

                def forward(self, x):
                    with torch.no_grad():
                        f = self.encoder(x)
                    return self.head(f.float())
            return LinearProbe()

        # multi-depth probe via timm's features_only / intermediate-layer API
        keep, is_plain_vit = _probe_indices(name)
        try:
            enc = timm.create_model(name, pretrained=pretrained, features_only=True,
                                    out_indices=tuple(keep))
            dims = enc.feature_info.channels()
        except (RuntimeError, AssertionError, KeyError, IndexError, TypeError):
            # models without features_only support: use forward_intermediates
            base = timm.create_model(name, pretrained=pretrained, num_classes=0)

            class Inter(nn.Module):
                def __init__(self):
                    super().__init__()
                    self.m = base

                def forward(self, x):
                    return list(self.m.forward_intermediates(
                        x, indices=keep, norm=True, intermediates_only=True))
            enc = Inter()
            dims = [base.num_features] * len(keep)
        return MultiLayerProbe(enc, n_classes, dims, hidden=probe_hidden)

    # full / lora
    kw = {}
    if mode == "full":
        kw["drop_path_rate"] = drop_path
    try:
        model = timm.create_model(name, pretrained=pretrained, num_classes=n_classes, **kw)
    except TypeError:
        model = timm.create_model(name, pretrained=pretrained, num_classes=n_classes)

    if mode == "lora":
        for p in model.parameters():
            p.requires_grad_(False)
        n = inject_lora(model, r=lora_r)
        if n == 0:
            raise ValueError(
                f"no LoRA target layers matched in {name}. LoRA here adapts attention "
                "projections (Linear or 1x1 Conv), so it fits transformers and hybrids "
                "but not pure CNNs such as ResNet, DenseNet, VGG or ConvNeXt -- for "
                "those use mode='probe' (frozen, strong) or mode='full'.")
        head = model.get_classifier()
        for p in head.parameters():
            p.requires_grad_(True)
        for mod in model.modules():         # norms are cheap and help a lot
            if isinstance(mod, (nn.LayerNorm, nn.BatchNorm2d)):
                for p in mod.parameters():
                    p.requires_grad_(True)
    return model


def count_params(model) -> tuple[int, int]:
    tot = sum(p.numel() for p in model.parameters())
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tot, tr


# ======================================================================================
# Two-stage transfer: ImageNet -> large FER corpus -> your ASD corpus
# ======================================================================================
STAGE2_RECIPE = """
TWO-STAGE TRANSFER  (do this before trying any new architecture)
================================================================
Expected payoff here is larger than any architecture swap on this list, and it is the
one experiment that produces a finding rather than a leaderboard row.

  Stage A  ImageNet weights                      (what you have now)
  Stage B  fine-tune on a large general FER corpus
  Stage C  fine-tune on your ASD corpus, subject-independent

For Stage B use AffectNet if you can obtain it -- a 2026 cross-scenario robustness study
found AffectNet-trained models generalised markedly better than RAF-DB-trained ones. If
AffectNet access is slow, RAF-DB or FER+ still beats going straight from ImageNet.

Map the label spaces explicitly and put the mapping in the paper. Your six classes are a
subset of the usual seven or eight; drop 'disgust' and 'contempt' from Stage B rather
than folding them into a nearby class.

Ablate all three: ImageNet-only vs +FER vs +FER+ASD. "Expression-domain pretraining is
worth X macro-F1 on ASD faces, and the gain is concentrated in the rare classes" is a
sentence a reviewer at JBHI or AIIM will find genuinely useful -- and it is a result
nobody has published for the ASD population.

One caution: check that your ASD sources do not overlap the FER corpus you pretrain on,
and say in the paper that you checked. Merged public face data has a habit of doing this.
"""


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--list", action="store_true")
    ap.add_argument("--tier", default=None)
    ap.add_argument("--check", default=None, help="model name to instantiate")
    ap.add_argument("--mode", default="full",
                    choices=["full", "lora", "probe", "linear"])
    ap.add_argument("--recipe", action="store_true")
    a = ap.parse_args()

    if a.recipe:
        print(STAGE2_RECIPE); return

    if a.list:
        for tier, desc in TIERS.items():
            if a.tier and tier != a.tier:
                continue
            rows = [e for e in MODEL_ZOO if e.tier == tier]
            if not rows:
                continue
            print(f"\n{'='*86}\n{tier.upper()}  —  {desc}\n{'='*86}")
            for e in rows:
                tag = "" if e.source == "timm" else f"  [{e.source}]"
                print(f"\n  {e.name}{tag}")
                for line in _wrap(e.why, 78):
                    print(f"      {line}")
                if e.note:
                    print(f"      → {e.note}")
        print()
        return

    if a.check:
        m = build_model(a.check, 6, mode=a.mode, pretrained=False)
        tot, tr = count_params(m)
        print(f"{a.check}  mode={a.mode}")
        print(f"  total params     {tot/1e6:8.2f} M")
        print(f"  trainable params {tr/1e6:8.2f} M  ({100*tr/max(tot,1):.2f}%)")
        x = torch.randn(2, 3, 224, 224)
        with torch.no_grad():
            y = m(x)
        print(f"  output shape     {tuple(y.shape)}")
        return

    ap.print_help()


def _wrap(s, w):
    out, line = [], ""
    for word in s.split():
        if len(line) + len(word) + 1 > w:
            out.append(line); line = word
        else:
            line = f"{line} {word}".strip()
    if line:
        out.append(line)
    return out


if __name__ == "__main__":
    main()